In [ ]:
import os
import re
import sqlite3
import logging
from typing import List, Tuple, Optional

import fitz as pymupdf  # PyMuPDF
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- Настройка логирования ----------
logging.getLogger("ollama").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- GPU для Ollama ----------
os.environ["OLLAMA_GPU_LAYERS"] = "-1"

# ---------- Глобальные переменные для NER-модели ----------
NER_MODEL_NAME = "bond005/rubert-multiconer"
ner_pipeline = None


def load_ner_pipeline():
    """Загружает пайплайн для NER. Вызывается один раз перед обработкой."""
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели: {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1,
        )
        print("NER-модель загружена.")
    return ner_pipeline


# ---------- Постобработка сущностей ----------
def clean_name(name: str) -> Optional[str]:
    """Очищает имя от мусора, оставшегося после NER/OCR. Возвращает None, если имя невалидно."""
    if not name:
        return None

    # Удаляем лишние пробелы
    name = " ".join(name.split())
    if not name:
        return None

    # Отбрасываем слишком короткие (например, одна буква)
    if len(name.split()) < 2:
        return None

    # Удаляем артефакты в начале/конце
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name, flags=re.IGNORECASE)
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+', ' ', name)

    # Исправляем типичные ошибки OCR
    fixes = {
        r'тов\.': 'товарищ',
        r'гр\-н': 'гражданин',
        r'проф\.': 'профессор',
    }
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)

    # Капитализация
    parts = name.split()
    parts = [p.capitalize() for p in parts]
    name = " ".join(parts)

    return name.strip() or None


def postprocess_entities(entities: List[Tuple[str, str]], source_file: str) -> List[Tuple[str, str, str]]:
    """
    entities: список (name, context)
    Возвращает: список (name, context, source_file) для вставки в БД
    """
    if not entities:
        return []

    cleaned_list = []
    seen_names = set()

    for raw_name, ctx in entities:
        name = clean_name(raw_name)
        if not name:
            continue

        # Пропускаем контекст-мусор
        if "===" in ctx or len(ctx.split()) < 3:
            continue

        if name not in seen_names:
            cleaned_list.append((name, ctx, source_file))
            seen_names.add(name)

    return cleaned_list


# ---------- Обработка текста с помощью RuBERT ----------
def process_and_store_rubert(text: str, db_path: str = "newspaper_analysis.db", source_file: str = ""):
    """Извлекает PER-сущности через RuBERT и сохраняет в БД."""
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    nlp = load_ner_pipeline()

    # Разбиваем текст на куски ~500 символов
    chunk_size = 500
    chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 10:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent["entity_group"] == "PER":
                    name = ent["word"]
                    # Примерный контекст
                    start_idx = chunk.find(name)
                    if start_idx == -1:
                        continue
                    ctx_start = max(0, start_idx - 80)
                    ctx_end = min(len(chunk), start_idx + len(name) + 80)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка при обработке куска: {e}")

    data_to_insert = postprocess_entities(raw_entities, source_file)

    if not data_to_insert:
        print("Имён людей не найдено")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS persons (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            context TEXT NOT NULL,
            source_file TEXT
        )
        """
    )
    cur.executemany(
        "INSERT INTO persons (name, context, source_file) VALUES (?, ?, ?)",
        data_to_insert,
    )
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data_to_insert)} записей в {db_path}")


# ---------- Обработка PDF ----------
def normalize_pdf_text(text: str) -> str:
    """Склеивает строки, разорванные внутри предложений, удаляет дефисы переноса."""
    # Склеиваем переносы с дефисами
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any(
            [
                not re.search(r"[.!?]$", prev),
                bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped, flags=re.IGNORECASE)),
                bool(re.search(r"[,;:\-—]$", prev)),
            ]
        )
        if continuation:
            cleaned[-1] = prev + " " + stripped
        else:
            cleaned.append(stripped)
    return "\n\n".join(cleaned)


# ---------- Исправление ошибок через LLM ----------
def fix_text_with_llm(chunk: str, model: str = "qwen2.5:7b") -> str:
    """Отправляет фрагмент локальной LLM для исправления OCR-ошибок."""
    prompt = f"""Ты — профессиональный корректор оцифрованных советских газет. Твоя задача — исправить только очевидные ошибки OCR, не меняя смысла, стиля, имён и чисел.

Правила:
1. Склей разорванные пробелами слова.
2. Замени украинские буквы на русские: 'і'→'и', 'ї'→'и', 'є'→'е', 'ґ'→'г'.
3. Исправь очевидные опечатки, но не меняй имена собственные без необходимости.
4. Удали мягкие переносы и лишние пробелы внутри слов.
5. НЕ трогай пунктуацию, числа, даты, сокращения типа "тов.", "проф.".

Исправь следующий фрагмент. Верни ТОЛЬКО исправленный текст, без пояснений.

Фрагмент:
{chunk}

Исправленный фрагмент:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "num_gpu": 999,
                "num_thread": 4,
            },
        )
        return response["response"].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk


def split_sentences(text: str) -> List[str]:
    """Разбивает текст на предложения по .!?"""
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def fix_text_advanced(text: str, method: str = "both", model: str = "qwen2.5:7b", show_progress: bool = True) -> str:
    """
    method: "regex", "llm", "both", "none"
    """
    # Базовая чистка регулярками
    text = re.sub(r"(\w)[­\-]\s+(\w)", r"\1\2", text)
    text = re.sub(r",\s*,", ",", text)
    text = re.sub(r"\.\s*\.", ".", text)

    if method == "regex":
        return text

    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text

        fixed_sentences = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process

        for sent in iterator:
            fixed = fix_text_with_llm(sent, model=model)
            fixed_sentences.append(fixed)

        result = []
        proc_idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed_sentences[proc_idx])
                proc_idx += 1
            else:
                result.append(sent)
        return " ".join(result)

    return text


def extract_text_from_pdf(
    pdf_path: str,
    fix_method: str = "both",
    model: str = "qwen2.5:7b",
    output_txt: str = "output.txt",
    analyze_with_ner: bool = True,
):
    """
    Извлекает текст из текстового PDF, нормализует, исправляет и анализирует.
    """
    print(f"Извлечение текста из: {pdf_path}")

    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение страниц PDF"):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    if fix_method != "none":
        full_text = fix_text_advanced(full_text, method=fix_method, model=model, show_progress=True)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"Текст сохранён в {output_txt}")

    if analyze_with_ner:
        process_and_store_rubert(full_text, source_file=pdf_path)


# ---------- Просмотр результатов ----------
def view_results(db_path: str = "newspaper_analysis.db", limit: int = 90):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, context, source_file FROM persons LIMIT ?", (limit,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("База данных пуста.")
        return
    for i, (name, ctx, src) in enumerate(rows, 1):
        print(f"{i}. Имя: {name}")
        print(f"   Контекст: {ctx[:500]}{'...' if len(ctx) > 500 else ''}")
        print(f"   Источник: {src}")
        print("-" * 60)


# ---------- Запуск ----------
if __name__ == "__main__":
    # ---------- НАСТРОЙКИ ----------
    pdf_file = r"C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf"

    # Метод исправления: "none", "regex", "llm", "both"
    FIX_METHOD = "both"
    # Модель Ollama
    LLM_MODEL = "qwen2.5:7b"

    # ---------- ВЫПОЛНЕНИЕ ----------
    extract_text_from_pdf(
        pdf_path=pdf_file,
        fix_method=FIX_METHOD,
        model=LLM_MODEL,
        output_txt="cleaned_text.txt",
    )

    # Показать сохранённые имена
    view_results()

ModuleNotFoundError: No module named 'ollama'

In [ ]:
import sqlite3

DB_PATH = "newspapers.db"

def print_all_mentions():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("SELECT id, newspaper_name, publication_date, person_name, context FROM newspaper_mentions")
    rows = cursor.fetchall()
    
    if not rows:
        print("Таблица пуста.")
        return
    
    # Заголовки
    print(f"{'ID':<4} {'Название газеты':<40} {'Дата выпуска':<12} {'Имя человека':<25} {'Контекст'}")
    print("-" * 110)
    
    for row in rows:
        id_, name, date, person, context = row
        # Обрезаем длинные строки для красивого вывода
        name = (name[:37] + '...') if len(name) > 40 else name
        person = (person[:22] + '...') if person and len(person) > 25 else (person or '')
        context = (context[:50] + '...') if context and len(context) > 50 else (context or '')
        print(f"{id_:<4} {name:<40} {date:<12} {person:<25} {context}")
    
    conn.close()
    print(f"\nВсего записей: {len(rows)}")

if __name__ == "__main__":
    print_all_mentions()

In [ ]:
import os
import re
import sqlite3
from typing import List, Tuple, Optional

import pymupdf
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- Настройки ----------
PDF_FOLDER = "downloaded_pdfs_pravda_kommunizma"
DB_PATH = "newspapers.db"          # Основная БД с таблицей newspaper_mentions
LLM_MODEL = "qwen2.5:7b"           # Модель Ollama для исправления OCR
FIX_METHOD = "both"                # "none", "regex", "llm", "both"
NER_MODEL_NAME = "bond005/rubert-multiconer"

# ---------- Глобальные NER ----------
ner_pipeline = None

def load_ner_pipeline():
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1
        )
        print("NER загружена.")
    return ner_pipeline

def clean_name(name: str) -> Optional[str]:
    if not name:
        return None
    name = ' '.join(name.split())
    if len(name.split()) < 2:
        return None
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name)
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name)
    name = re.sub(r'\s+', ' ', name)
    fixes = {r'тов\.': 'товарищ', r'гр\-н': 'гражданин', r'проф\.': 'профессор'}
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)
    parts = name.split()
    name = ' '.join(p.capitalize() for p in parts)
    return name.strip()

def postprocess_entities(entities: List[Tuple[str, str]], source_file: str) -> List[Tuple[str, str, str]]:
    cleaned = []
    seen = set()
    for raw_name, ctx in entities:
        name = clean_name(raw_name)
        if not name or "===" in ctx or len(ctx.split()) < 3:
            continue
        if name not in seen:
            cleaned.append((name, ctx, source_file))
            seen.add(name)
    return cleaned

def process_and_store_rubert(text: str, newspaper_name: str, publication_date: str, pdf_filename: str):
    """Извлекает PER-сущности и сохраняет их в newspaper_mentions."""
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    nlp = load_ner_pipeline()
    chunk_size = 500
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 10:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent['entity_group'] == 'PER':
                    name = ent['word']
                    start = chunk.find(name)
                    if start == -1:
                        continue
                    ctx_start = max(0, start - 80)
                    ctx_end = min(len(chunk), start + len(name) + 80)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка NER: {e}")

    data = postprocess_entities(raw_entities, pdf_filename)
    if not data:
        print("Имён не найдено")
        return

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # Убедимся, что таблица имеет нужные поля (person_name, context, pdf_filename)
    cur.execute('''
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    ''')
    # Вставляем каждое упоминание
    for name, ctx, pdf_fname in data:
        cur.execute('''
            INSERT INTO newspaper_mentions (newspaper_name, publication_date, person_name, context, pdf_filename)
            VALUES (?, ?, ?, ?, ?)
        ''', (newspaper_name, publication_date, name, ctx, pdf_fname))
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data)} упоминаний в {DB_PATH}")

def normalize_pdf_text(text: str) -> str:
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

def fix_text_with_llm(chunk: str, model: str = LLM_MODEL) -> str:
    prompt = f"""Ты — корректор советских газет. Исправь только очевидные ошибки OCR.
Правила:
- Склей разорванные слова: "изв ест ня ко в о го" -> "известнякового"
- Замени украинские буквы: і→и, ї→и, є→е, ґ→г
- Исправь опечатки: "коммуннст" -> "коммунист"
- Удали мягкие переносы
- НЕ меняй пунктуацию, числа, даты, сокращения (тов., проф.)

Фрагмент:
{chunk}

Исправленный фрагмент:"""
    try:
        response = ollama.generate(model=model, prompt=prompt, options={"num_gpu": 999, "num_thread": 4})
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = FIX_METHOD, model: str = LLM_MODEL, show_progress=True) -> str:
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)
    if method == "regex":
        return text
    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text
        fixed = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process
        for sent in iterator:
            fixed.append(fix_text_with_llm(sent, model))
        result = []
        idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed[idx])
                idx += 1
            else:
                result.append(sent)
        return ' '.join(result)
    return text

def extract_date_and_name_from_filename(filename: str) -> Tuple[Optional[str], Optional[str]]:
    """Из имени файла YYYY-MM-DD_Название.pdf возвращает (дата, название)."""
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split('_', 1)
    if len(parts) == 2 and re.match(r'\d{4}-\d{2}-\d{2}', parts[0]):
        return parts[0], parts[1].replace('_', ' ')
    return None, None

def process_pdf_file(pdf_path: str):
    print(f"\nОбработка: {pdf_path}")
    pdf_filename = os.path.basename(pdf_path)
    # Пропускаем, если уже обработан (проверка по pdf_filename в БД)
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM newspaper_mentions WHERE pdf_filename = ? LIMIT 1", (pdf_filename,))
    if cur.fetchone():
        print(f"  Уже обработан, пропускаем.")
        conn.close()
        return
    conn.close()

    date, name = extract_date_and_name_from_filename(pdf_path)
    if not date or not name:
        print(f"  Не удалось извлечь дату/название из имени файла, пропускаем.")
        return

    # Извлечение текста
    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение PDF", leave=False):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    if FIX_METHOD != "none":
        full_text = fix_text_advanced(full_text, method=FIX_METHOD, model=LLM_MODEL, show_progress=True)

    # Анализ и сохранение
    process_and_store_rubert(full_text, newspaper_name=name, publication_date=date, pdf_filename=pdf_filename)

def process_all_pdfs_in_folder(folder: str):
    pdf_files = [f for f in os.listdir(folder) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"В папке {folder} нет PDF-файлов.")
        return
    print(f"Найдено PDF: {len(pdf_files)}")
    for pdf_file in pdf_files:
        process_pdf_file(os.path.join(folder, pdf_file))

if __name__ == "__main__":
    process_all_pdfs_in_folder(PDF_FOLDER)

ModuleNotFoundError: No module named 'ollama'

In [ ]:
import os
import re
import sqlite3
from typing import List, Tuple, Optional

import pymupdf
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- НАСТРОЙКИ ----------
PDF_FOLDER = "downloaded_pdfs_pravda_kommunizma"   # папка с PDF-файлами
DB_PATH = "newspapers.db"                         # путь к БД (та же, что в db_creator)
LLM_MODEL = "qwen2.5:7b"                          # модель Ollama для исправления OCR
FIX_METHOD = "both"                               # "none", "regex", "llm", "both"
NER_MODEL_NAME = "bond005/rubert-multiconer"      # модель для распознавания имён

# ---------- Глобальная переменная для NER (загружается один раз) ----------
ner_pipeline = None

# ---------- Функции работы с БД ----------
def ensure_table_exists():
    """Проверяет существование таблицы newspaper_mentions и создаёт её при необходимости."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)
    conn.commit()
    conn.close()

# ---------- Загрузка NER-модели ----------
def load_ner_pipeline():
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1
        )
        print("NER-модель загружена.")
    return ner_pipeline

# ---------- Очистка и постобработка имён ----------
def clean_name(name: str) -> Optional[str]:
    if not name:
        return None
    name = ' '.join(name.split())
    if len(name.split()) < 2:          # слишком короткие имена отбрасываем
        return None
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name)
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name)
    name = re.sub(r'\s+', ' ', name)
    fixes = {r'тов\.': 'товарищ', r'гр\-н': 'гражданин', r'проф\.': 'профессор'}
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)
    parts = name.split()
    name = ' '.join(p.capitalize() for p in parts)
    return name.strip()

def postprocess_entities(entities: List[Tuple[str, str]], source_file: str) -> List[Tuple[str, str, str]]:
    cleaned = []
    seen = set()
    for raw_name, ctx in entities:
        name = clean_name(raw_name)
        if not name or "===" in ctx or len(ctx.split()) < 3:
            continue
        if name not in seen:
            cleaned.append((name, ctx, source_file))
            seen.add(name)
    return cleaned

# ---------- NER + сохранение в БД ----------
def process_and_store_rubert(text: str, newspaper_name: str, publication_date: str, pdf_filename: str):
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    nlp = load_ner_pipeline()
    chunk_size = 500
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 10:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent['entity_group'] == 'PER':
                    name = ent['word']
                    start = chunk.find(name)
                    if start == -1:
                        continue
                    ctx_start = max(0, start - 80)
                    ctx_end = min(len(chunk), start + len(name) + 80)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка NER: {e}")

    data = postprocess_entities(raw_entities, pdf_filename)
    if not data:
        print("Имён не найдено")
        return

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    for name, ctx, pdf_fname in data:
        cur.execute("""
            INSERT INTO newspaper_mentions (newspaper_name, publication_date, person_name, context, pdf_filename)
            VALUES (?, ?, ?, ?, ?)
        """, (newspaper_name, publication_date, name, ctx, pdf_fname))
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data)} упоминаний в {DB_PATH}")

# ---------- Нормализация текста PDF ----------
def normalize_pdf_text(text: str) -> str:
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

# ---------- Исправление OCR через LLM ----------
def fix_text_with_llm(chunk: str, model: str = LLM_MODEL) -> str:
    prompt = f"""Ты — корректор советских газет. Исправь только очевидные ошибки OCR.
Правила:
- Склей разорванные слова: "изв ест ня ко в о го" -> "известнякового"
- Замени украинские буквы: і→и, ї→и, є→е, ґ→г
- Исправь опечатки: "коммуннст" -> "коммунист"
- Удали мягкие переносы
- НЕ меняй пунктуацию, числа, даты, сокращения (тов., проф.)

Фрагмент:
{chunk}

Исправленный фрагмент:"""
    try:
        response = ollama.generate(model=model, prompt=prompt, options={"num_gpu": 999, "num_thread": 4})
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = FIX_METHOD, model: str = LLM_MODEL, show_progress=True) -> str:
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)
    if method == "regex":
        return text
    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text
        fixed = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process
        for sent in iterator:
            fixed.append(fix_text_with_llm(sent, model))
        result = []
        idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed[idx])
                idx += 1
            else:
                result.append(sent)
        return ' '.join(result)
    return text

# ---------- Извлечение даты и названия из имени файла ----------
def extract_date_and_name_from_filename(filename: str) -> Tuple[Optional[str], Optional[str]]:
    """Ожидается формат: ГГГГ-ММ-ДД_Название газеты.pdf"""
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split('_', 1)
    if len(parts) == 2 and re.match(r'\d{4}-\d{2}-\d{2}', parts[0]):
        return parts[0], parts[1].replace('_', ' ')
    return None, None

# ---------- Обработка одного PDF ----------
def process_pdf_file(pdf_path: str):
    print(f"\nОбработка: {pdf_path}")
    pdf_filename = os.path.basename(pdf_path)

    ensure_table_exists()   # гарантируем наличие таблицы

    # Пропускаем, если уже есть записи с таким pdf_filename
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM newspaper_mentions WHERE pdf_filename = ? LIMIT 1", (pdf_filename,))
    if cur.fetchone():
        print(f"  Уже обработан, пропускаем.")
        conn.close()
        return
    conn.close()

    date, name = extract_date_and_name_from_filename(pdf_path)
    if not date or not name:
        print(f"  Не удалось извлечь дату/название из имени файла, пропускаем.")
        return

    # Извлечение текста из PDF
    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение PDF", leave=False):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    if FIX_METHOD != "none":
        full_text = fix_text_advanced(full_text, method=FIX_METHOD, model=LLM_MODEL, show_progress=True)

    process_and_store_rubert(full_text, newspaper_name=name, publication_date=date, pdf_filename=pdf_filename)

# ---------- Обработка всех PDF в папке ----------
def process_all_pdfs_in_folder(folder: str):
    ensure_table_exists()
    pdf_files = [f for f in os.listdir(folder) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"В папке {folder} нет PDF-файлов.")
        return
    print(f"Найдено PDF: {len(pdf_files)}")
    for pdf_file in pdf_files:
        process_pdf_file(os.path.join(folder, pdf_file))

# ---------- Точка входа ----------
if __name__ == "__main__":
    process_all_pdfs_in_folder(PDF_FOLDER)

ModuleNotFoundError: No module named 'transformers'

In [ ]:
import os
import re
import sqlite3
from typing import List, Tuple, Optional

import pymupdf
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- НАСТРОЙКИ ----------
PDF_FOLDER = "downloaded_pdfs_pravda_kommunizma"
DB_PATH = "newspapers.db"
LLM_MODEL = "qwen2.5:7b"
FIX_METHOD = "both"                # "none", "regex", "llm", "both"
NER_MODEL_NAME = "bond005/rubert-multiconer"

# ---------- Глобальная NER ----------
ner_pipeline = None

# ---------- ФУНКЦИЯ, КОТОРАЯ ГАРАНТИРУЕТ НАЛИЧИЕ ТАБЛИЦЫ (ДОБАВЛЕНО) ----------
def ensure_table_exists():
    """Создаёт таблицу newspaper_mentions, если её ещё нет."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)
    conn.commit()
    conn.close()

# ---------- ВАШ ИСХОДНЫЙ КОД (БЕЗ ИЗМЕНЕНИЙ, КРОМЕ ВСТАВКИ ensure_table_exists) ----------

def load_ner_pipeline():
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1
        )
        print("NER-модель загружена.")
    return ner_pipeline

def clean_name(name: str) -> Optional[str]:
    if not name:
        return None
    name = ' '.join(name.split())
    if len(name.split()) < 2:
        return None
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name)
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name)
    name = re.sub(r'\s+', ' ', name)
    fixes = {r'тов\.': 'товарищ', r'гр\-н': 'гражданин', r'проф\.': 'профессор'}
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)
    parts = name.split()
    name = ' '.join(p.capitalize() for p in parts)
    return name.strip()

def postprocess_entities(entities: List[Tuple[str, str]], source_file: str) -> List[Tuple[str, str, str]]:
    cleaned = []
    seen = set()
    for raw_name, ctx in entities:
        name = clean_name(raw_name)
        if not name or "===" in ctx or len(ctx.split()) < 3:
            continue
        if name not in seen:
            cleaned.append((name, ctx, source_file))
            seen.add(name)
    return cleaned

def process_and_store_rubert(text: str, newspaper_name: str, publication_date: str, pdf_filename: str):
    """Извлекает PER-сущности и сохраняет их в newspaper_mentions."""
    ensure_table_exists()  # <-- ДОБАВЛЕНО: гарантия наличия таблицы

    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    nlp = load_ner_pipeline()
    chunk_size = 1000
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 10:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent['entity_group'] == 'PER':
                    name = ent['word']
                    start = chunk.find(name)
                    if start == -1:
                        continue
                    ctx_start = max(0, start - 80)
                    ctx_end = min(len(chunk), start + len(name) + 80)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка NER: {e}")

    data = postprocess_entities(raw_entities, pdf_filename)
    if not data:
        print("Имён не найдено")
        return

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    for name, ctx, pdf_fname in data:
        cur.execute("""
            INSERT INTO newspaper_mentions (newspaper_name, publication_date, person_name, context, pdf_filename)
            VALUES (?, ?, ?, ?, ?)
        """, (newspaper_name, publication_date, name, ctx, pdf_fname))
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data)} упоминаний в {DB_PATH}")

def normalize_pdf_text(text: str) -> str:
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

def fix_text_with_llm(chunk: str, model: str = LLM_MODEL) -> str:
    prompt = f"""Ты — корректор советских газет. Исправь только очевидные ошибки OCR.
Правила:
- Склей разорванные слова: "изв ест ня ко в о го" -> "известнякового"
- Замени украинские буквы: і→и, ї→и, є→е, ґ→г
- Исправь опечатки: "коммуннст" -> "коммунист"
- Удали мягкие переносы
- НЕ меняй пунктуацию, числа, даты, сокращения (тов., проф.)

Фрагмент:
{chunk}

Исправленный фрагмент:"""
    try:
        response = ollama.generate(model=model, prompt=prompt, options={"num_gpu": 999, "num_thread": 4})
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = FIX_METHOD, model: str = LLM_MODEL, show_progress=True) -> str:
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)
    if method == "regex":
        return text
    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text
        fixed = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process
        for sent in iterator:
            fixed.append(fix_text_with_llm(sent, model))
        result = []
        idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed[idx])
                idx += 1
            else:
                result.append(sent)
        return ' '.join(result)
    return text

def extract_date_and_name_from_filename(filename: str) -> Tuple[Optional[str], Optional[str]]:
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split('_', 1)
    if len(parts) == 2 and re.match(r'\d{4}-\d{2}-\d{2}', parts[0]):
        return parts[0], parts[1].replace('_', ' ')
    return None, None

def process_pdf_file(pdf_path: str):
    print(f"\nОбработка: {pdf_path}")
    pdf_filename = os.path.basename(pdf_path)

    # Проверка, обработан ли уже (опционально)
    ensure_table_exists()
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM newspaper_mentions WHERE pdf_filename = ? LIMIT 1", (pdf_filename,))
    if cur.fetchone():
        print(f"  Уже обработан, пропускаем.")
        conn.close()
        return
    conn.close()

    date, name = extract_date_and_name_from_filename(pdf_path)
    if not date or not name:
        print(f"  Не удалось извлечь дату/название из имени файла, пропускаем.")
        return

    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение PDF", leave=False):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    if FIX_METHOD != "none":
        full_text = fix_text_advanced(full_text, method=FIX_METHOD, model=LLM_MODEL, show_progress=True)

    process_and_store_rubert(full_text, newspaper_name=name, publication_date=date, pdf_filename=pdf_filename)

def process_all_pdfs_in_folder(folder: str):
    pdf_files = [f for f in os.listdir(folder) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"В папке {folder} нет PDF-файлов.")
        return
    print(f"Найдено PDF: {len(pdf_files)}")
    for pdf_file in pdf_files:
        process_pdf_file(os.path.join(folder, pdf_file))

if __name__ == "__main__":
    process_all_pdfs_in_folder(PDF_FOLDER)

Найдено PDF: 2

Обработка: downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf


LLM исправление: 100%|██████████| 447/447 [05:34<00:00,  1.34предл./s]


Загрузка NER-модели bond005/rubert-multiconer...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7817.94it/s]


NER-модель загружена.


NER (RuBERT): 100%|██████████| 44/44 [00:06<00:00,  7.21chunk/s]


Сохранено 23 упоминаний в newspapers.db

Обработка: downloaded_pdfs_pravda_kommunizma\1980-07-31_Правда коммунизма 1980 091.pdf


LLM исправление:  84%|████████▍ | 473/560 [05:01<00:55,  1.57предл./s]


KeyboardInterrupt: 

In [ ]:
import os
import re
import sqlite3
import logging
from typing import List, Tuple, Optional

import pymupdf  # PyMuPDF
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- Настройка логирования ----------
logging.getLogger("ollama").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- GPU для Ollama ----------
os.environ["OLLAMA_GPU_LAYERS"] = "-1"

# ---------- Глобальные переменные для NER-модели (загружаются один раз) ----------
NER_MODEL_NAME = "bond005/rubert-multiconer"
ner_pipeline = None

def load_ner_pipeline():
    """Загружает пайплайн для NER. Вызывается один раз перед обработкой."""
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели: {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1  # автоопределение GPU
        )
        print("NER-модель загружена.")
    return ner_pipeline

# ---------- Постобработка сущностей ----------
def clean_name(name: str) -> Optional[str]:
    """
    Очищает имя от мусора, оставшегося после NER/OCR.
    Возвращает None, если имя невалидно.
    """
    if not name:
        return None

    # Удаляем лишние пробелы и приводим к нормальному регистру (первая буква заглавная)
    name = ' '.join(name.split())
    if not name:
        return None

    # Отбрасываем слишком короткие
    if len(name.split()) < 2:
        return None

    # Удаляем артефакты в начале/конце
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name)      # кавычки, скобки в начале
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name)      # в конце
    name = re.sub(r'\s+', ' ', name)

    # Исправляем типичные ошибки OCR (расширяемый словарь)
    fixes = {
        r'тов\.': 'товарищ',
        r'гр\-н': 'гражданин',
        r'проф\.': 'профессор',
    }
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)

    # Капитализация: первая буква каждого слова заглавная (кроме служебных)
    parts = name.split()
    new_parts = []
    for i, p in enumerate(parts):
        # Не делаем заглавными предлоги/частицы в середине, но для простоты капитализируем всё
        new_parts.append(p.capitalize())
    name = ' '.join(new_parts)

    return name.strip()

def postprocess_entities(entities: List[Tuple[str, str]], source_file: str) -> List[Tuple[str, str, str]]:
    """
    entities: список (name, context)
    Возвращает: список (name, context, source_file) для вставки в БД
    """
    if not entities:
        return []

    cleaned_list = []
    seen_names = set()

    for raw_name, ctx in entities:
        name = clean_name(raw_name)
        if not name:
            continue

        # Пропускаем контекст-мусор
        if "===" in ctx or len(ctx.split()) < 3:
            continue

        if name not in seen_names:
            cleaned_list.append((name, ctx, source_file))
            seen_names.add(name)

    return cleaned_list

# ---------- Обработка текста с помощью RuBERT ----------
def process_and_store_rubert(text: str, db_path: str = "newspaper_analysis.db", source_file: str = ""):
    """Извлекает PER-сущности через RuBERT и сохраняет в БД."""
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    nlp = load_ner_pipeline()

    # Разбиваем текст на куски ~500 символов (чтобы не превысить лимит модели)
    chunk_size = 500
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 10:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent['entity_group'] == 'PER':
                    name = ent['word']
                    # Находим примерный контекст
                    start_idx = chunk.find(name)
                    if start_idx == -1:
                        continue
                    ctx_start = max(0, start_idx - 80)
                    ctx_end = min(len(chunk), start_idx + len(name) + 80)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка при обработке куска: {e}")

    data_to_insert = postprocess_entities(raw_entities, source_file)

    if not data_to_insert:
        print("Имён людей не найдено")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS persons (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            context TEXT NOT NULL,
            source_file TEXT
        )
    ''')
    cur.executemany(
        "INSERT INTO persons (name, context, source_file) VALUES (?, ?, ?)",
        data_to_insert
    )
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data_to_insert)} записей в {db_path}")

# ---------- Обработка PDF ----------
def normalize_pdf_text(text: str) -> str:
    """Склеивает строки, разорванные внутри предложений, удаляет дефисы переноса."""
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

# ---------- Исправление ошибок через LLM (улучшенный промпт) ----------
def fix_text_with_llm(chunk: str, model: str = "qwen2.5:7b") -> str:
    """Отправляет фрагмент локальной LLM для исправления OCR-ошибок."""
    prompt = f"""Ты — профессиональный корректор оцифрованных советских газет. Твоя задача — исправить только очевидные ошибки OCR, не меняя смысла, стиля, имён и чисел.

**Правила:**
1. Склей разорванные пробелами слова: "изв ест ня ко в о го" → "известнякового".
2. Замени украинские буквы на русские: 'і'→'и', 'ї'→'и', 'є'→'е', 'ґ'→'г'.
3. Исправь очевидные опечатки: "горняіЛ" → "горняк", "коммуннст" → "коммунист".
4. Удали мягкие переносы и лишние пробелы внутри слов: "ме­ таллург" → "металлург".
5. НЕ трогай пунктуацию, числа, даты, сокращения типа "тов.", "проф.".

**Примеры:**
Вход:  "комумунист Владимир Леонидович Карпенікюв"
Выход: "коммунист Владимир Леонидович Карпеников"

Вход:  "изв ест ня ко в о го ка рь ера Александр Иванович Чер­ ных"
Выход: "известнякового карьера Александр Иванович Черных"

Вход:  "Звание «Почетный горняіЛ» получил бурильщик тов. Петров"
Выход: "Звание «Почетный горняк» получил бурильщик тов. Петров"

Вход:  "Секретарь обкома ВКП(б) т. Смирнов"
Выход: "Секретарь обкома ВКП(б) т. Смирнов"   (сокращение не меняем)

Исправь следующий фрагмент. Верни ТОЛЬКО исправленный текст, без пояснений.

Фрагмент:
{chunk}

Исправленный фрагмент:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "num_gpu": 999,
                "num_thread": 4
            }
        )
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> List[str]:
    """Разбивает текст на предложения по .!?"""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = "both", model: str = "qwen2.5:7b", show_progress: bool = True) -> str:
    """
    method: "regex", "llm", "both", "none"
    """
    # Базовая чистка регулярками
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)

    if method == "regex":
        return text

    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text

        fixed_sentences = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process

        for sent in iterator:
            fixed = fix_text_with_llm(sent, model=model)
            fixed_sentences.append(fixed)

        result = []
        proc_idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed_sentences[proc_idx])
                proc_idx += 1
            else:
                result.append(sent)
        return ' '.join(result)

    return text

def extract_text_from_pdf(pdf_path: str,
                          fix_method: str = "both",
                          model: str = "qwen2.5:7b",
                          output_txt: str = "output.txt",
                          analyze_with_ner: bool = True):
    """
    Извлекает текст из текстового PDF, нормализует, исправляет и анализирует.
    """
    print(f"Извлечение текста из: {pdf_path}")

    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение страниц PDF"):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    if fix_method != "none":
        full_text = fix_text_advanced(full_text, method=fix_method, model=model, show_progress=True)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"Текст сохранён в {output_txt}")

    if analyze_with_ner:
        process_and_store_rubert(full_text, source_file=pdf_path)

# ---------- Просмотр результатов ----------
def view_results(db_path: str = "newspaper_analysis.db", limit: int = 100):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, context, source_file FROM persons LIMIT ?", (limit,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("База данных пуста.")
        return
    for i, (name, ctx, src) in enumerate(rows, 1):
        print(f"{i}. Имя: {name}")
        print(f"   Контекст: {ctx[:500]}{'...' if len(ctx)>500 else ''}")
        print(f"   Источник: {src}")
        print("-" * 60)

# ---------- ЗАПУСК ----------
if __name__ == "__main__":
    # ---------- НАСТРОЙКИ ----------
    pdf_file = r"C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf"

    # Метод исправления: "none", "regex", "llm", "both"
    FIX_METHOD = "both"
    # Модель Ollama (рекомендую qwen2.5:7b или saiga2:7b)
    LLM_MODEL = "qwen2.5:7b"

    # ---------- ВЫПОЛНЕНИЕ ----------
    extract_text_from_pdf(
        pdf_path=pdf_file,
        fix_method=FIX_METHOD,
        model=LLM_MODEL,
        output_txt="cleaned_text.txt"
    )

    # Показать сохранённые имена
    view_results()

c:\Users\nikol\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Извлечение текста из: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf


LLM исправление: 100%|██████████| 447/447 [05:11<00:00,  1.43предл./s]


Текст сохранён в cleaned_text.txt
Загрузка NER-модели: bond005/rubert-multiconer...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7846.75it/s]


NER-модель загружена.


NER (RuBERT): 100%|██████████| 83/83 [00:06<00:00, 12.35chunk/s]

Сохранено 10 записей в newspaper_analysis.db
1. Имя: И. Хохлов
   Контекст: елами предприятия. С докладом о состоянии дел в цехе выступил начальник цеха О. И. Хохлов. В ходе беседы рабочие поднимали многие вопросы производства и быта, вносили пр
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
2. Имя: Д Ильич Брежnev
   Контекст: д Ильич Брежnev отмечал: «Советский человек — хозяин своей страны. Он — советский человек — еди
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
3. Имя: Верой Але
   Контекст: Алые паруса», молодогвардейцы, «Тимуровец». Совместно с методистом Ж КО поселка Верой Але
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
---------------------

In [ ]:
import os
import re
import sqlite3
import logging
from typing import List, Tuple, Optional

import pymupdf  # PyMuPDF
import ollama
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ---------- Настройка логирования ----------
logging.getLogger("ollama").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- GPU для Ollama ----------
os.environ["OLLAMA_GPU_LAYERS"] = "-1"

# ---------- Глобальные переменные для NER-модели (загружаются один раз) ----------
NER_MODEL_NAME = "bond005/rubert-multiconer"
ner_pipeline = None

def load_ner_pipeline():
    """Загружает пайплайн для NER. Вызывается один раз перед обработкой."""
    global ner_pipeline
    if ner_pipeline is None:
        print(f"Загрузка NER-модели: {NER_MODEL_NAME}...")
        tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
        model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
        ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1
        )
        print("NER-модель загружена.")
    return ner_pipeline

# ---------- Очистка текста от мусора (оставляем ТОЛЬКО русские символы) ----------
def clean_russian_only(text: str) -> str:
    """
    Оставляет только русские буквы (кириллица), цифры, пробелы и базовую пунктуацию.
    Удаляет латиницу, иероглифы, спецсимволы.
    """
    if not text:
        return ""
    
    # Разрешённые символы:
    # а-яА-ЯёЁ - русские буквы
    # 0-9 - цифры
    # \s - пробельные символы
    # .,!?;:()-«»"№% - пунктуация и спецзнаки
    allowed_pattern = re.compile(r'[^а-яА-ЯёЁ0-9\s\.\,\!\?\;\:\(\)\-«»\"\'№\%]')
    
    # Заменяем все запрещённые символы на пробел
    text = allowed_pattern.sub(' ', text)
    
    # Убираем множественные пробелы
    text = re.sub(r'\s+', ' ', text)
    
    # Удаляем пробелы в начале и конце
    text = text.strip()
    
    return text

# ---------- Исправление ошибок через LLM ----------
def fix_text_with_llm(chunk: str, model: str = "qwen2.5:7b") -> str:
    """Отправляет фрагмент локальной LLM для исправления OCR-ошибок."""
    prompt = f"""Ты — профессиональный корректор оцифрованных советских газет. Твоя задача — исправить только очевидные ошибки OCR, не меняя смысла, стиля, имён и чисел.

Правила:
1. Склей разорванные пробелами слова: "изв ест ня ко в о го" → "известнякового".
2. Исправь очевидные опечатки: "горняіЛ" → "горняк", "коммуннст" → "коммунист".
3. Удали мягкие переносы и лишние пробелы внутри слов: "ме­ таллург" → "металлург".
4. НЕ трогай пунктуацию, числа, даты, сокращения типа "тов.", "проф.".

Примеры:
Вход:  "комумунист Владимир Леонидович Карпенікюв"
Выход: "коммунист Владимир Леонидович Карпеников"

Вход:  "изв ест ня ко в о го ка рь ера Александр Иванович Чер­ ных"
Выход: "известнякового карьера Александр Иванович Черных"

Вход:  "Звание «Почетный горняіЛ» получил бурильщик тов. Петров"
Выход: "Звание «Почетный горняк» получил бурильщик тов. Петров"

Исправь следующий фрагмент. Верни ТОЛЬКО исправленный текст, без пояснений.

Фрагмент:
{chunk}

Исправленный фрагмент:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "num_gpu": 999,
                "num_thread": 4
            }
        )
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

# ---------- Постобработка сущностей ----------
def clean_name(name: str) -> Optional[str]:
    """
    Очищает имя от мусора, оставшегося после NER/OCR.
    Возвращает None, если имя невалидно.
    """
    if not name:
        return None

    # Сначала очищаем от всех не-русских символов
    name = clean_russian_only(name)
    
    # Удаляем лишние пробелы
    name = ' '.join(name.split())
    if not name:
        return None

    # Отбрасываем слишком короткие
    if len(name.split()) < 2:
        return None

    # Удаляем артефакты в начале/конце
    name = re.sub(r'^[^\wА-ЯЁ]+', '', name)
    name = re.sub(r'[^\wА-ЯЁ]+$', '', name)
    name = re.sub(r'\s+', ' ', name)

    # Исправляем типичные сокращения
    fixes = {
        r'тов\.': 'товарищ',
        r'гр\-н': 'гражданин',
        r'проф\.': 'профессор',
    }
    for pat, repl in fixes.items():
        name = re.sub(pat, repl, name, flags=re.IGNORECASE)

    # Капитализация
    parts = name.split()
    new_parts = [p.capitalize() for p in parts]
    name = ' '.join(new_parts)

    return name.strip()

def postprocess_entities(entities: List[Tuple[str, str]], source_file: str, llm_model: Optional[str] = None) -> List[Tuple[str, str, str]]:
    """
    entities: список (name, context)
    llm_model: если указана, контекст дополнительно исправляется через LLM
    Возвращает: список (name, context, source_file) для вставки в БД
    """
    if not entities:
        return []

    cleaned_list = []
    seen_names = set()

    for raw_name, ctx in entities:
        # Дополнительное исправление контекста через LLM (если нужно)
        if llm_model and len(ctx) > 10:
            try:
                ctx = fix_text_with_llm(ctx, model=llm_model)
            except Exception as e:
                print(f"Ошибка LLM при обработке контекста: {e}")
                pass  # Если LLM недоступна, оставляем как есть
        
        # Очищаем от не-русских символов
        ctx = clean_russian_only(ctx)
        name = clean_name(raw_name)
        
        if not name:
            continue

        # Пропускаем слишком короткий контекст
        if len(ctx.split()) < 3:
            continue

        if name not in seen_names:
            cleaned_list.append((name, ctx, source_file))
            seen_names.add(name)

    return cleaned_list

# ---------- Обработка текста с помощью RuBERT ----------
def process_and_store_rubert(text: str, db_path: str = "newspaper_analysis.db", 
                             source_file: str = "", llm_model: Optional[str] = None):
    """Извлекает PER-сущности через RuBERT и сохраняет в БД."""
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    # Финальная очистка перед NER
    text = clean_russian_only(text)
    
    nlp = load_ner_pipeline()

    # Разбиваем текст на куски ~500 символов
    chunk_size = 700
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    raw_entities = []
    for chunk in tqdm(chunks, desc="NER (RuBERT)", unit="chunk"):
        if len(chunk) < 1:
            continue
        try:
            results = nlp(chunk)
            for ent in results:
                if ent['entity_group'] == 'PER':
                    name = ent['word']
                    # Находим примерный контекст
                    start_idx = chunk.find(name)
                    if start_idx == -1:
                        continue
                    ctx_start = max(0, start_idx - 100)
                    ctx_end = min(len(chunk), start_idx + len(name) + 100)
                    ctx = chunk[ctx_start:ctx_end].strip()
                    raw_entities.append((name, ctx))
        except Exception as e:
            print(f"Ошибка при обработке куска: {e}")

    data_to_insert = postprocess_entities(raw_entities, source_file, llm_model)

    if not data_to_insert:
        print("Имён людей не найдено")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS persons (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            context TEXT NOT NULL,
            source_file TEXT
        )
    ''')
    cur.executemany(
        "INSERT INTO persons (name, context, source_file) VALUES (?, ?, ?)",
        data_to_insert
    )
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data_to_insert)} записей в {db_path}")

# ---------- Обработка PDF ----------
def normalize_pdf_text(text: str) -> str:
    """Склеивает строки, разорванные внутри предложений, удаляет дефисы переноса."""
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-я0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

def split_sentences(text: str) -> List[str]:
    """Разбивает текст на предложения по .!?"""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = "both", model: str = "qwen2.5:7b", show_progress: bool = True) -> str:
    """
    method: "regex", "llm", "both", "none"
    """
    # Базовая чистка регулярками
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)

    if method == "regex":
        return text

    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text

        fixed_sentences = []
        iterator = tqdm(to_process, desc="LLM исправление", unit="предл.") if show_progress else to_process

        for sent in iterator:
            fixed = fix_text_with_llm(sent, model=model)
            fixed_sentences.append(fixed)

        result = []
        proc_idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed_sentences[proc_idx])
                proc_idx += 1
            else:
                result.append(sent)
        return ' '.join(result)

    return text

def extract_text_from_pdf(pdf_path: str,
                          fix_method: str = "both",
                          model: str = "qwen2.5:7b",
                          output_txt: str = "output.txt",
                          analyze_with_ner: bool = True):
    """
    Извлекает текст из текстового PDF, нормализует, исправляет и анализирует.
    """
    print(f"Извлечение текста из: {pdf_path}")

    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение страниц PDF"):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    # ПЕРВИЧНАЯ ОЧИСТКА - оставляем только русские символы
    full_text = clean_russian_only(full_text)
    
    if fix_method != "none":
        full_text = fix_text_advanced(full_text, method=fix_method, model=model, show_progress=True)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"Текст сохранён в {output_txt}")

    if analyze_with_ner:
        process_and_store_rubert(full_text, source_file=pdf_path, llm_model=model)

# ---------- Просмотр результатов ----------
def view_results(db_path: str = "newspaper_analysis.db", limit: int = 100):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, context, source_file FROM persons LIMIT ?", (limit,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("База данных пуста.")
        return
    for i, (name, ctx, src) in enumerate(rows, 1):
        print(f"{i}. Имя: {name}")
        print(f"   Контекст: {ctx[:500]}{'...' if len(ctx)>500 else ''}")
        print(f"   Источник: {src}")
        print("-" * 60)

# ---------- Очистка БД (опционально) ----------
def clear_database(db_path: str = "newspaper_analysis.db"):
    """Очищает таблицу persons в БД."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("DELETE FROM persons")
    conn.commit()
    conn.close()
    print(f"База данных {db_path} очищена.")

# ---------- ЗАПУСК ----------
if __name__ == "__main__":
    # ---------- НАСТРОЙКИ ----------
    pdf_file = r"C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf"

    # Метод исправления: "none", "regex", "llm", "both"
    FIX_METHOD = "both"
    # Модель Ollama (рекомендую qwen2.5:7b или saiga2:7b)
    LLM_MODEL = "qwen2.5:7b"

    # Опционально: очистить БД перед запуском
    # clear_database()

    # ---------- ВЫПОЛНЕНИЕ ----------
    extract_text_from_pdf(
        pdf_path=pdf_file,
        fix_method=FIX_METHOD,
        model=LLM_MODEL,
        output_txt="cleaned_text.txt"
    )

    # Показать сохранённые имена
    view_results()

Извлечение текста из: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf


LLM исправление: 100%|██████████| 447/447 [05:24<00:00,  1.38предл./s]


Текст сохранён в cleaned_text.txt
Загрузка NER-модели: bond005/rubert-multiconer...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8240.28it/s]


NER-модель загружена.


NER (RuBERT): 100%|██████████| 60/60 [00:05<00:00, 10.26chunk/s]


Сохранено 4 записей в newspaper_analysis.db
1. Имя: И. Хохлов
   Контекст: елами предприятия. С докладом о состоянии дел в цехе выступил начальник цеха О. И. Хохлов. В ходе беседы рабочие поднимали многие вопросы производства и быта, вносили пр
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
2. Имя: Д Ильич Брежnev
   Контекст: д Ильич Брежnev отмечал: «Советский человек — хозяин своей страны. Он — советский человек — еди
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
3. Имя: Верой Але
   Контекст: Алые паруса», молодогвардейцы, «Тимуровец». Совместно с методистом Ж КО поселка Верой Але
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
----------------------

In [ ]:
import os
import re
import sqlite3
import logging
import pymupdf  # PyMuPDF
import ollama
from tqdm import tqdm
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding,
    NewsNERTagger, Doc
)

# ---------- Настройка логирования ----------
# Подавляем информационные сообщения от ollama (HTTP Request POST ... 200 OK)
logging.getLogger("ollama").setLevel(logging.WARNING)

# ---------- GPU для Ollama ----------
os.environ["OLLAMA_GPU_LAYERS"] = "-1"   # загрузить все слои модели на GPU

# ---------- Наташа (NER) ----------
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

def postprocess_entities(entities: list, source_file: str) -> list:
    """
    Удаляет дубликаты, неполные имена и служебный контекст.
    Вход: список кортежей (person_name, context_sentence)
    Выход: список (name, context, source_file) для вставки в БД
    """
    if not entities:
        return []

    # Группируем по контексту (одно предложение)
    by_ctx = {}
    for name, ctx in entities:
        if ctx not in by_ctx:
            by_ctx[ctx] = []
        by_ctx[ctx].append(name)

    cleaned = []
    for ctx, names in by_ctx.items():
        # Пропускаем контекст, похожий на заголовки страниц или слишком короткий
        if "===" in ctx or len(ctx.split()) < 3:
            continue

        # Внутри одного контекста берём самое длинное имя (полное ФИО)
        best_name = sorted(names, key=len)[-1] if len(names) > 1 else names[0]

        # Отбрасываем однословные имена
        if len(best_name.split()) < 2:
            continue

        cleaned.append((best_name, ctx))

    # Убираем глобальные дубликаты имён
    final, seen = [], set()
    for name, ctx in cleaned:
        if name not in seen:
            final.append((name, ctx, source_file))
            seen.add(name)

    return final

def process_and_store(text: str, db_path: str = "newspaper_analysis.db", source_file: str = ""):
    """Находит имена людей (PER) через Natasha и сохраняет в SQLite."""
    if not text or not text.strip():
        print("Нет текста для анализа")
        return

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    sents = list(doc.sents)
    raw_entities = []

    for span in doc.spans:
        if span.type == "PER":
            span.normalize(morph_vocab)
            name = span.normal if span.normal else span.text

            # Найти предложение, содержащее этот span
            ctx = ""
            for s in sents:
                if s.start <= span.start < s.stop:
                    ctx = s.text.strip()
                    break
            ctx = ctx or span.text
            raw_entities.append((name, ctx))

    data_to_insert = postprocess_entities(raw_entities, source_file)

    if not data_to_insert:
        print("Имён людей не найдено")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS persons (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            context TEXT NOT NULL,
            source_file TEXT
        )
    ''')
    cur.executemany(
        "INSERT INTO persons (name, context, source_file) VALUES (?, ?, ?)",
        data_to_insert
    )
    conn.commit()
    conn.close()
    print(f"Сохранено {len(data_to_insert)} записей в {db_path}")

# ---------- Обработка текста ----------
def normalize_pdf_text(text: str) -> str:
    """Склеивает строки, разорванные внутри предложений, удаляет дефисы переноса."""
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)  # склейка переносов
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        # Признаки, что текущая строка — продолжение предыдущей
        continuation = any([
            not re.search(r'[.!?]$', prev),                     # нет точки в конце
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),  # начинается со строчной
            bool(re.search(r'[,;:\-—]$', prev)),                # заканчивается на запятую и т.п.
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

# ---------- Исправление ошибок через LLM (Ollama) ----------
def fix_text_with_llm(chunk: str, model: str = "qwen2:7b") -> str:
    """Отправляет фрагмент локальной LLM для исправления очевидных опечаток."""
    prompt = f"""Ты — корректор текста после OCR. Исправь только явные ошибки, не меняя смысл, стиль, имена собственные и числа.

Правила:
1. Склей слова, разорванные пробелами: "изв ест ня ко в о го" → "известнякового"
2. Замени украинские буквы на русские: 'і'→'и', 'ї'→'и', 'є'→'е', 'ґ'→'г'
3. Исправь очевидные опечатки: "горняіЛ" → "горняк", "прамоты" → "грамоты"
4. Удали лишние пробелы внутри слов: "ме­ таллург" → "металлург"
5. НЕ трогай пунктуацию, числа, даты.

Примеры:
Вход:  "комумунист Владимир Леонидович Карпенікюв"
Выход: "коммунист Владимир Леонидович Карпеников"

Вход:  "изв ест ня ко в о го ка рь ера Александр Иванович Чер­ ных"
Выход: "известнякового карьера Александр Иванович Черных"

Вход:  "Звание «Почетный горняіЛ» получил бурильщик"
Выход: "Звание «Почетный горняк» получил бурильщик"

Исправь следующий текст. Верни ТОЛЬКО исправленный текст, без пояснений.

Текст:
{chunk}

Исправленный текст:"""
    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "num_gpu": 999,   # максимум слоёв на GPU
                "num_thread": 4
            }
        )
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> list:
    """Разбивает текст на предложения по .!?"""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = "regex", model: str = "qwen2:7b", show_progress: bool = True) -> str:
    """
    method:
        - "regex": только регулярные выражения (быстро)
        - "llm":   исправление каждого предложения через LLM
        - "both":  сначала regex, потом LLM
        - "none":  без исправлений
    show_progress: показывать прогресс-бар с оценкой оставшегося времени
    """
    # Лёгкая очистка регулярками
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)  # мягкий перенос
    text = re.sub(r',\s*,', ',', text)                 # двойные запятые
    text = re.sub(r'\.\s*\.', '.', text)               # двойные точки

    if method == "regex":
        return text

    if method in ("llm", "both"):
        sentences = split_sentences(text)
        # Фильтруем слишком короткие предложения, которые не будем обрабатывать LLM
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text

        fixed_sentences = []
        # Используем tqdm для отображения прогресса и оставшегося времени
        if show_progress:
            iterator = tqdm(to_process, desc="Исправление текста (LLM)", unit="предл.")
        else:
            iterator = to_process

        for sent in iterator:
            fixed = fix_text_with_llm(sent, model=model)
            fixed_sentences.append(fixed)

        # Собираем обратно, сохраняя короткие предложения как есть
        result = []
        proc_idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed_sentences[proc_idx])
                proc_idx += 1
            else:
                result.append(sent)
        return ' '.join(result)

    return text

def extract_text_from_pdf(pdf_path: str,
                          fix_method: str = "both",
                          model: str = "qwen2:7b",
                          output_txt: str = "output.txt",
                          analyze_with_natasha: bool = True):
    """
    Извлекает текст из текстового PDF, нормализует, исправляет и анализирует.
    """
    print(f"Извлечение текста из: {pdf_path}")

    doc = pymupdf.open(pdf_path)
    full_text = ""
    # Прогресс по страницам
    for page in tqdm(doc, desc="Чтение страниц PDF"):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    # Дополнительное исправление (если включено)
    if fix_method != "none":
        full_text = fix_text_advanced(full_text, method=fix_method, model=model, show_progress=True)

    # Сохраняем в текстовый файл
    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"Текст сохранён в {output_txt}")

    # Анализ имён
    if analyze_with_natasha:
        process_and_store(full_text, source_file=pdf_path)

# ---------- Просмотр результатов ----------
def view_results(db_path: str = "newspaper_analysis.db", limit: int = 100):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, context, source_file FROM persons LIMIT ?", (limit,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("База данных пуста.")
        return
    for i, (name, ctx, src) in enumerate(rows, 1):
        print(f"{i}. Имя: {name}")
        print(f"   Контекст: {ctx[:500]}{'...' if len(ctx)>500 else ''}")
        print(f"   Источник: {src}")
        print("-" * 60)

# ---------- ЗАПУСК ----------
if __name__ == "__main__":
    # Укажите путь к вашему PDF
    pdf_file = r"C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf"

    # Метод исправления: "none", "regex", "llm", "both"
    extract_text_from_pdf(
        pdf_path=pdf_file,
        fix_method="both",      # качественная очистка
        model="qwen2:7b"        # можно заменить на qwen2.5:7b
    )

    # Показать сохранённые имена
    view_results()

Извлечение текста из: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf


Исправление текста (LLM): 100%|██████████| 447/447 [04:53<00:00,  1.52предл./s]


Текст сохранён в output.txt
Сохранено 55 записей в newspaper_analysis.db
1. Имя: И. Хохлов
   Контекст: елами предприятия. С докладом о состоянии дел в цехе выступил начальник цеха О. И. Хохлов. В ходе беседы рабочие поднимали многие вопросы производства и быта, вносили пр
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
2. Имя: Д Ильич Брежnev
   Контекст: д Ильич Брежnev отмечал: «Советский человек — хозяин своей страны. Он — советский человек — еди
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 088.pdf
------------------------------------------------------------
3. Имя: Верой Але
   Контекст: Алые паруса», молодогвардейцы, «Тимуровец». Совместно с методистом Ж КО поселка Верой Але
   Источник: C:\Users\nikol\Desktop\disser\downloaded_pdfs_pravda_kommunizma\1980-07-24_Правда коммунизма 1980 0

In [ ]:
import os
import re
import sqlite3
import logging
from pathlib import Path

import pymupdf
import ollama
from tqdm import tqdm
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding,
    NewsNERTagger, Doc
)

# ---------- Настройка логирования ----------
logging.getLogger("ollama").setLevel(logging.WARNING)

# ---------- GPU для Ollama ----------
os.environ["OLLAMA_GPU_LAYERS"] = "-1"

# ---------- Инициализация Natasha ----------
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

# ---------- Вспомогательные функции (из первого кода) ----------
def normalize_pdf_text(text: str) -> str:
    """Склеивает строки, разорванные внутри предложений, удаляет дефисы переноса."""
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if not cleaned:
            cleaned.append(stripped)
            continue
        prev = cleaned[-1]
        continuation = any([
            not re.search(r'[.!?]$', prev),
            bool(re.match(r'^[а-яa-z0-9»"\')\]},;:.\-]', stripped)),
            bool(re.search(r'[,;:\-—]$', prev)),
        ])
        if continuation:
            cleaned[-1] = prev + ' ' + stripped
        else:
            cleaned.append(stripped)
    return '\n\n'.join(cleaned)

def fix_text_with_llm(chunk: str, model: str = "qwen2:7b") -> str:
    """Исправляет опечатки через Ollama."""
    prompt = f"""Ты — корректор текста после OCR. Исправь только явные ошибки, не меняя смысл, стиль, имена собственные (исключение-явные опечатки) и числа.

Правила:
1. Склей слова, разорванные пробелами: "изв ест ня ко в о го" → "известнякового"
2. Замени украинские буквы на русские: 'і'→'и', 'ї'→'и', 'є'→'е', 'ґ'→'г'
3. Исправь очевидные опечатки: "горняіЛ" → "горняк", "прамоты" → "грамоты"
4. Удали лишние пробелы внутри слов: "ме­ таллург" → "металлург"
5. НЕ трогай пунктуацию, числа, даты.

Исправь следующий текст. Верни ТОЛЬКО исправленный текст, БЕЗ ПОЯСНЕНИЙ И КОММЕНТАРИЕВ.

Текст:
{chunk}

Исправленный текст:"""
    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={"num_gpu": 999, "num_thread": 4}
        )
        return response['response'].strip()
    except Exception as e:
        print(f"Ошибка LLM: {e}")
        return chunk

def split_sentences(text: str) -> list:
    """Разбивает текст на предложения."""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def fix_text_advanced(text: str, method: str = "both", model: str = "qwen2:7b", show_progress: bool = True) -> str:
    """Исправление текста: regex, llm или оба."""
    text = re.sub(r'(\w)[­\-]\s+(\w)', r'\1\2', text)
    text = re.sub(r',\s*,', ',', text)
    text = re.sub(r'\.\s*\.', '.', text)

    if method == "regex":
        return text

    if method in ("llm", "both"):
        sentences = split_sentences(text)
        to_process = [s for s in sentences if len(s) > 10]
        if not to_process:
            return text

        fixed_sentences = []
        iterator = tqdm(to_process, desc="Исправление текста (LLM)", unit="предл.") if show_progress else to_process
        for sent in iterator:
            fixed = fix_text_with_llm(sent, model=model)
            fixed_sentences.append(fixed)

        result = []
        proc_idx = 0
        for sent in sentences:
            if len(sent) > 10:
                result.append(fixed_sentences[proc_idx])
                proc_idx += 1
            else:
                result.append(sent)
        return ' '.join(result)
    return text

def postprocess_entities(entities: list, source_file: str) -> list:
    """Очистка и дедупликация имен."""
    if not entities:
        return []
    by_ctx = {}
    for name, ctx in entities:
        by_ctx.setdefault(ctx, []).append(name)

    cleaned = []
    for ctx, names in by_ctx.items():
        if "===" in ctx or len(ctx.split()) < 3:
            continue
        best_name = sorted(names, key=len)[-1] if len(names) > 1 else names[0]
        if len(best_name.split()) < 2:
            continue
        cleaned.append((best_name, ctx))

    final, seen = [], set()
    for name, ctx in cleaned:
        if name not in seen:
            final.append((name, ctx, source_file))
            seen.add(name)
    return final

def extract_names_from_text(text: str, source_file: str) -> list:
    """Находит имена людей через Natasha и возвращает список (name, context, source_file)."""
    if not text or not text.strip():
        return []

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    sents = list(doc.sents)
    raw_entities = []

    for span in doc.spans:
        if span.type == "PER":
            span.normalize(morph_vocab)
            name = span.normal if span.normal else span.text
            ctx = ""
            for s in sents:
                if s.start <= span.start < s.stop:
                    ctx = s.text.strip()
                    break
            ctx = ctx or span.text
            raw_entities.append((name, ctx))

    return postprocess_entities(raw_entities, source_file)

# ---------- Работа с БД newspaper_mentions ----------
DB_PATH = "newspapers.db"

def init_database():
    """Создаёт таблицу newspaper_mentions, если её нет."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)
    conn.commit()
    conn.close()
    print(f"База данных '{DB_PATH}' готова.")

def save_mentions(mentions: list):
    """Сохраняет список кортежей (newspaper_name, publication_date, person_name, context, pdf_filename) в БД."""
    if not mentions:
        return
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.executemany("""
        INSERT INTO newspaper_mentions (newspaper_name, publication_date, person_name, context, pdf_filename)
        VALUES (?, ?, ?, ?, ?)
    """, mentions)
    conn.commit()
    conn.close()
    print(f"Сохранено {len(mentions)} упоминаний.")

# ---------- Парсинг имени файла ----------
def parse_filename(filename: str) -> tuple:
    """
    Извлекает дату и название газеты из имени файла.
    Ожидается формат: YYYY-MM-DD_Название.pdf
    Возвращает (date_str, newspaper_name).
    """
    match = re.match(r'^(\d{4}-\d{2}-\d{2})_(.+)\.pdf$', filename, re.IGNORECASE)
    if match:
        date_str = match.group(1)
        newspaper_name = match.group(2).strip()
        return date_str, newspaper_name
    else:
        # fallback: используем имя файла без расширения как название, дата неизвестна
        name = Path(filename).stem
        return "0000-00-00", name

# ---------- Обработка одного PDF ----------
def process_pdf_file(pdf_path: str, fix_method: str = "both", model: str = "qwen2:7b"):
    """Извлекает текст, находит имена и сохраняет в newspapers.db."""
    print(f"\nОбработка: {pdf_path}")
    filename = os.path.basename(pdf_path)
    pub_date, newspaper_name = parse_filename(filename)

    # Извлечение текста
    doc = pymupdf.open(pdf_path)
    full_text = ""
    for page in tqdm(doc, desc="Чтение страниц", leave=False):
        raw = page.get_text()
        full_text += normalize_pdf_text(raw) + "\n"
    doc.close()

    # Исправление текста (опционально)
    if fix_method != "none":
        full_text = fix_text_advanced(full_text, method=fix_method, model=model, show_progress=False)

    # Поиск имен
    names_data = extract_names_from_text(full_text, pdf_path)
    if not names_data:
        print("Имён не найдено.")
        return

    # Подготовка записей для БД
    mentions = []
    for person_name, context, _ in names_data:
        mentions.append((
            newspaper_name,
            pub_date,
            person_name,
            context,
            filename
        ))

    save_mentions(mentions)

# ---------- Обработка всей папки ----------
def process_folder(folder_path: str, fix_method: str = "both", model: str = "qwen2:7b"):
    """Обрабатывает все PDF-файлы в указанной папке."""
    if not os.path.isdir(folder_path):
        print(f"Папка не найдена: {folder_path}")
        return

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print("Нет PDF-файлов в папке.")
        return

    print(f"Найдено {len(pdf_files)} PDF-файлов.")
    init_database()

    for pdf_file in tqdm(pdf_files, desc="Общий прогресс", unit="файл"):
        full_path = os.path.join(folder_path, pdf_file)
        try:
            process_pdf_file(full_path, fix_method=fix_method, model=model)
        except Exception as e:
            print(f"Ошибка при обработке {pdf_file}: {e}")

# ---------- Просмотр результатов ----------
def view_results(limit: int = 100):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        SELECT newspaper_name, publication_date, person_name, context, pdf_filename
        FROM newspaper_mentions LIMIT ?
    """, (limit,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("База данных пуста.")
        return
    for i, (newspaper, date, name, ctx, fname) in enumerate(rows, 1):
        print(f"{i}. Газета: {newspaper} ({date})")
        print(f"   Имя: {name}")
        print(f"   Контекст: {ctx[:300]}{'...' if len(ctx)>300 else ''}")
        print(f"   Файл: {fname}")
        print("-" * 60)

# ---------- ЗАПУСК ----------
if __name__ == "__main__":
    MAGAZINES_FOLDER = r"C:\Users\nikol\Desktop\disser\magazines"

    process_folder(
        folder_path=MAGAZINES_FOLDER,
        fix_method="both",      # "none", "regex", "llm", "both"
        model="qwen2:7b"
    )

    # Показать сохранённые упоминания
    view_results()

Найдено 10 PDF-файлов.
База данных 'newspapers.db' готова.


Общий прогресс:   0%|          | 0/10 [00:00<?, ?файл/s]


Обработка: C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf


Общий прогресс:  10%|█         | 1/10 [06:24<57:39, 384.41s/файл]

Сохранено 63 упоминаний.

Обработка: C:\Users\nikol\Desktop\disser\magazines\1980-07-19_Правда коммунизма 1980 086.pdf


...

Сохранено 63 упоминаний.

Обработка: C:\Users\nikol\Desktop\disser\magazines\1980-09-03_Правда коммунизма 1980 106.pdf


Общий прогресс:  90%|█████████ | 9/10 [47:59<04:44, 284.32s/файл]

Сохранено 48 упоминаний.

Обработка: C:\Users\nikol\Desktop\disser\magazines\1980-09-13_Правда коммунизма 1980 114.pdf


Общий прогресс: 100%|██████████| 10/10 [53:31<00:00, 321.19s/файл]

Сохранено 70 упоминаний.
1. Газета: Правда коммунизма 1980 081 (1980-07-08)
   Имя: В. Д. Конева
   Контекст: Звено выполняет свой план на 110% и перевершает В. Д. Конева), новагралях, Михаил совхоза "Глинский" НабыMitаЕвич МитаЕв и Сергей увели темпы.
   Файл: 1980-07-08_Правда коммунизма 1980 081.pdf
------------------------------------------------------------
2. Газета: Правда коммунизма 1980 081 (1980-07-08)
   Имя: Николая ПавлоДмитриевич Дмитриев
   Контекст: Заготовкой корМихайлович Ермаков - стомов здесь занимается комгометамй, Иван Ёвдокимоплексное звено под рукович Дудкин и Анатолий водством Николая ПавлоДмитриевич Дмитриев — вича Епифанова.
   Файл: 1980-07-08_Правда коммунизма 1980 081.pdf
------------------------------------------------------------
3. Газета: Правда коммунизма 1980 081 (1980-07-08)
   Имя: Федорович Бачинин
   Контекст: Но каждый выполняет порученное дедого из шести членов зведо с душой ро стараниеМ на свои обязанности: Леопомня, что темпы и качестнид Федо

In [ ]:
import sqlite3

DB_PATH = "newspapers.db"

def show_all_records():
    """Выводит все записи из таблицы newspaper_mentions."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id, newspaper_name, publication_date, person_name, context, pdf_filename
        FROM newspaper_mentions
        ORDER BY publication_date DESC, id
    """)
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        print("База данных пуста.")
        return

    print(f"\n{'='*80}")
    print(f"НАЙДЕНО ЗАПИСЕЙ: {len(rows)}")
    print(f"{'='*80}\n")

    for i, (row_id, newspaper, pub_date, person, context, pdf_file) in enumerate(rows, 1):
        print(f"{i}. ID: {row_id}")
        print(f"   Газета: {newspaper}")
        print(f"   Дата: {pub_date}")
        print(f"   Имя: {person}")
        print(f"   Контекст: {context[:500]}{'...' if len(context) > 500 else ''}")
        print(f"   Файл: {pdf_file}")
        print("-" * 80)

if __name__ == "__main__":
    show_all_records()


НАЙДЕНО ЗАПИСЕЙ: 10

1. ID: 5
   Газета: Правда коммунизма. 1980. № 114
   Дата: 1980-09-13
   Имя: None


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import pymupdf  # pip install pymupdf

def get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text

# Пример:
text = get_text_from_pdf(r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf")
print(text)

ПРОЯКТАММ tc ix  СТ»АЯ.СОВДНН«ИТ1СЫ
ОРГАН РЕЖЕВСКОГО 
ГОРОДСКОГО КОМИТЕТА 
КПСС И РЕЖЕВСКОГО 
ГОРОДСКОГО СОВЕТА 
НАРОДНЫХ ДЕПУТАТОВ коммунизме
ВТОРНИК,
8 ИЮЛЯ 1980 г.
СМсТРЧ / ) Ь  
_
с ѵ
П *«с 1 им.
но 81 (6543)
o S '4 L S o
°  
Т У Т
С Т Р А Д Н Ы Е  
П РО Ц ЕН ТЫ
Сенокосная страда в Ара- работает на 
подборщике, 
маншовском отделении (уп- Сергей Алексеевич 
Горбу- центов.
ЗЕЛЕНАЯ ЖАТВА
лотая пора сенокоса 
оп­
ределена строгими грани­
цами. 
Звено выполняет 
свой план на 110 
про-
равдяющий В. Д. Конев), нов — на граблях, Михаил 
совхоза «Глинский» наби- Митаевич Митаев и Сергей 
рает темпы. Заготовкой кор- Михайлович Ермаков— сто- 
мов здесь занимается ком- гометамй, Иван Ёвдокимо- 
плексное звено под 
руко- вич Дудкин и 
Анатолий 
водством Николая 
Павло- Дмитриевич Дмитриев —
вича Епифанова. 
У каж- скирдоправами. Но каждый
выполняет порученное де- 
дого из шести членов зве- до с душой> ро стараниеМ(
на свои обязанности: Лео- помня, что темпы и качест- 
нид Федорович 
Бачинин во 

In [ ]:
import pymupdf
import ollama

def fix_text_with_llm(text: str, model: str = "qwen2:7b") -> str:
    prompt = f"""Исправь опечатки и склей разорванные слова в тексте. 
    Верни только исправленный текст, без пояснений.
    
    Текст:
    {text}
    """
    response = ollama.generate(model=model, prompt=prompt)
    return response['response'].strip()

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = []

    for page in doc:
        # вариант: "text" даёт одну строку, близкую к "plain"
        page_text = page.get_text("text")
        raw_pages.append(page_text)

    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)

# Использование
pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
raw_text = get_text_from_pdf(pdf_path)
fixed_text = fix_text_with_llm(raw_text)
print(fixed_text)

ModuleNotFoundError: No module named 'ollama'

In [ ]:
import pymupdf
import re

def clean_pdf_text(text: str) -> str:
    # 1. Убираем переносы слов с мягким переносом (висячий дефис + разрыв строки)
    #    Пример: "обеспечивают­\nся" → "обеспечиваются"
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    
    # 2. Склеиваем строки, которые не заканчиваются знаком конца предложения
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    
    # 3. Объединяем обратно с пробелами
    cleaned = '\n'.join(merged)
    
    # 4. Убираем лишние пробелы вокруг знаков препинания (опционально)
    cleaned = re.sub(r'\s+([.,!?;:])', r'\1', cleaned)
    cleaned = re.sub(r'([„"«(])\s+', r'\1', cleaned)
    cleaned = re.sub(r'\s+([»"»)]),', r'\1', cleaned)
    
    return cleaned

def get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

# Пример использования:
text = get_text_from_pdf(r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf")
print(text)

ПРОЯКТАММ tc ix  СТ»АЯ.СОВДНН«ИТ1СЫ ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС И РЕЖЕВСКОГО ГОРОДСКОГО СОВЕТА НАРОДНЫХ ДЕПУТАТОВ коммунизме ВТОРНИК, 8 ИЮЛЯ 1980 г.
СМсТРЧ / ) Ь _ с ѵ П *«с 1 им.
но 81 (6543) o S '4 L S o ° Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов.
ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами.
Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы. Заготовкой кор- Михайлович Ермаков— сто- мов здесь занимается ком- гометамй, Иван Ёвдокимо- плексное звено под руко- вич Дудкин и Анатолий водством Николая Павло- Дмитриевич Дмитриев — вича Епифанова.
У каж- скирдоправами. Но каждый выполняет порученное де- дого из шести членов зве- до с душой> ро стараниеМ(на свои обязанности: Лео- помня, что темпы и качест- нид Федорович Бачинин во сейчас главное — зо- ВПЕРЕД И—ПЕРВ

In [ ]:
import pymupdf
import re
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc

def clean_pdf_text(text: str) -> str:
    """Базовая чистка: склейка строк и мягких переносов"""
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    cleaned = '\n'.join(merged)
    cleaned = re.sub(r'\s+([.,!?;:])', r'\1', cleaned)
    cleaned = re.sub(r'([„"«(])\s+', r'\1', cleaned)
    cleaned = re.sub(r'\s+([»"»)]),', r'\1', cleaned)
    return cleaned

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

def extract_names_with_context(text: str):
    """Находит имена людей (PER) и возвращает список (имя, предложение_контекст)"""
    segmenter = Segmenter()
    emb = NewsEmbedding()
    ner_tagger = NewsNERTagger(emb)
    
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)
    
    # Разбиваем на предложения (для поиска контекста)
    sentences = list(doc.sents)
    results = []
    
    for span in doc.spans:
        if span.type == "PER":
            # Находим предложение, содержащее этот span
            context = ""
            for sent in sentences:
                if sent.start <= span.start < sent.stop:
                    context = sent.text.strip()
                    break
            if not context:
                context = span.text
            results.append((span.text, context))
    
    # Убираем дубликаты (имя + одинаковый контекст)
    unique = list(dict.fromkeys(results))
    return unique

# Пример использования
if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    
    print("Извлечение имён людей...\n")
    names_contexts = extract_names_with_context(full_text)
    
    for name, ctx in names_contexts:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx)>200 else ''}")
        print("-" * 70)
    
    print(f"\nВсего найдено уникальных упоминаний: {len(names_contexts)}")

Извлечение имён людей...

Имя: Сергей Алексеевич Горбу-
Контекст: ПРОЯКТАММ tc ix  СТ»АЯ.СОВДНН«ИТ1СЫ ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС И РЕЖЕВСКОГО ГОРОДСКОГО СОВЕТА НАРОДНЫХ ДЕПУТАТОВ коммунизме ВТОРНИК, 8 ИЮЛЯ 1980 г.
СМсТРЧ / ) Ь _ с ѵ П *«с 1 им.
но 81 ...
----------------------------------------------------------------------
Имя: Звено
Контекст: Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы.
----------------------------------------------------------------------
Имя: В. Д. Конев
Контекст: Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы.
----------------------------------------------------------------------
Имя: Михаил
Контекст: Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы.


In [ ]:
import pymupdf
import re
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc

def advanced_clean_text(text: str) -> str:
    """Расширенная чистка текста перед NER"""
    # 1. Убираем переносы слов с дефисом (в том числе мягкий перенос)
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # 2. Склеиваем строки, которые не являются концом предложения
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    text = ' '.join(merged)  # теперь весь текст в одной строке (с пробелами)
    
    # 3. Исправляем типичные OCR-ошибки с пунктуацией
    text = re.sub(r'(\w)\s*,\s*(\w)', r'\1, \2', text)      # пробелы вокруг запятой
    text = re.sub(r'(\w)\s*\.\s*(\w)', r'\1. \2', text)     # пробелы вокруг точки
    text = re.sub(r',\s*,', ',', text)                       # двойные запятые
    text = re.sub(r'\.\s*\.', '.', text)                     # двойные точки
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)             # убираем пробелы перед знаками
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)            # один пробел после знака
    
    # 4. Исправляем украинские буквы и другие частые замены
    replacements = {
        'і': 'и', 'ї': 'и', 'є': 'е', 'ґ': 'г',
        'ъ': 'ь', 'Ъ': 'Ь', 'ё': 'е',
        '№': '№',  # оставляем как есть
        '«': '"', '»': '"', '“': '"', '”': '"',
    }
    for wrong, right in replacements.items():
        text = text.replace(wrong, right)
    
    # 5. Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def fix_name(name: str) -> str:
    """Очищает имя от мусора и восстанавливает инициалы"""
    # Убираем обрамляющие символы
    name = re.sub(r'^[\s\-–—.,;:]+|[\s\-–—.,;:]+$', '', name)
    # Исправляем "Л, А." -> "Л. А."
    name = re.sub(r'(\w)\s*,\s*(\w)', r'\1. \2.', name)
    # Убираем лишние точки в инициалах: "А.." -> "А."
    name = re.sub(r'\.{2,}', '.', name)
    # Если имя начинается с маленькой буквы, делаем заглавной
    if name and name[0].islower():
        name = name[0].upper() + name[1:]
    return name

def extract_names_with_context(text: str, window_left: int = 60, window_right: int = 150):
    """Ищет имена и вырезает контекстное окно вокруг них"""
    segmenter = Segmenter()
    emb = NewsEmbedding()
    ner_tagger = NewsNERTagger(emb)
    
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)
    
    results = []
    for span in doc.spans:
        if span.type != "PER":
            continue
        name = span.text
        # Очистка имени
        name_clean = fix_name(name)
        # Фильтруем мусорные имена
        if len(name_clean) < 3:
            continue
        if re.fullmatch(r'[А-Я]\.?(\s[А-Я]\.?)?', name_clean):
            continue
        if re.search(r'\d', name_clean):
            continue
        
        # Вырезаем контекст: окно вокруг span по символьным позициям
        start = max(0, span.start_char - window_left)
        end = min(len(text), span.stop_char + window_right)
        context = text[start:end].strip()
        # Убираем начало с многоточием (если есть)
        context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
        results.append((name_clean, context))
    
    # Дедупликация
    unique = []
    seen = set()
    for name, ctx in results:
        key = (name, ctx[:100])
        if key not in seen:
            seen.add(key)
            unique.append((name, ctx))
    return unique

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return advanced_clean_text(text)

# Пример использования
if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    
    print("Извлечение имён с контекстом...\n")
    names_contexts = extract_names_with_context(full_text)
    
    for name, ctx in names_contexts:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx)>200 else ''}")
        print("-" * 70)
    
    print(f"\nВсего уникальных упоминаний: {len(names_contexts)}")

Извлечение имён с контекстом...



AttributeError: 'DocSpan' object has no attribute 'start_char'

In [ ]:
import pymupdf
import re
import spacy

# Загрузка русской модели spaCy (предварительно установить: python -m spacy download ru_core_news_sm)
nlp = spacy.load("ru_core_news_sm")

def clean_pdf_text(text: str) -> str:
    """Улучшенная чистка текста: склейка строк, удаление переносов, исправление пробелов"""
    # Убираем мягкие переносы (висячий дефис)
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # Склеиваем строки, не являющиеся концом предложения
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    text = ' '.join(merged)  # сплошной текст
    
    # Исправляем типичные OCR-ошибки с пунктуацией
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)   # пробелы перед знаками
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)  # один пробел после знака
    text = re.sub(r'(\w)\s*,\s*(\w)', r'\1, \2', text)
    text = re.sub(r'(\w)\s*\.\s*(\w)', r'\1. \2', text)
    
    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_text_from_pdf(pdf_path: str) -> str:
    """Извлекает текст из PDF и чистит его"""
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

def extract_names_with_spacy(text: str, window_left: int = 60, window_right: int = 200):
    """
    Находит имена (PER) с помощью spaCy и возвращает список (имя, контекст)
    Контекст — это окно символов вокруг имени.
    """
    doc = nlp(text)
    results = []
    
    for ent in doc.ents:
        if ent.label_ == "PER":
            name = ent.text.strip()
            # Фильтрация мусорных имён
            if len(name) < 2:
                continue
            if re.fullmatch(r'[А-Я]\.?(\s[А-Я]\.?)?', name):  # одиночные инициалы
                continue
            if re.search(r'\d', name):
                continue
            
            # Вырезаем контекстное окно (по символам)
            start = max(0, ent.start_char - window_left)
            end = min(len(text), ent.end_char + window_right)
            context = text[start:end].strip()
            # Убираем начальный мусор (например, "...")
            context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
            results.append((name, context))
    
    # Дедупликация по имени + началу контекста
    unique = []
    seen = set()
    for name, ctx in results:
        key = (name, ctx[:100])
        if key not in seen:
            seen.add(key)
            unique.append((name, ctx))
    return unique

# Пример использования
if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    
    print("Извлечение текста из PDF...")
    full_text = get_text_from_pdf(pdf_path)
    
    print("Поиск имён людей (spaCy)...")
    names_contexts = extract_names_with_spacy(full_text)
    
    print(f"\nНайдено уникальных упоминаний: {len(names_contexts)}\n")
    for name, ctx in names_contexts[:100]:  # выводим первые 20
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:500]}{'...' if len(ctx) > 500 else ''}")
        print("-" * 70)

c:\Users\nikol\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Извлечение текста из PDF...
Поиск имён людей (spaCy)...

Найдено уникальных упоминаний: 136

Имя: Т Р А Д Н Ы Е П
Контекст: / ) Ь _ с ѵ П *«с 1 им. но 81 (6543) o S '4 L S o ° Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов. ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами. Звено выполняет
----------------------------------------------------------------------
Имя: Сергей Алексеевич Горбу-
Контекст: да в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов. ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами. Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев
----------------------------------------------------------------------
Имя: В. Д. Конев
Контекст: ми границами. Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на граблях, Михаил совхоза 

In [ ]:
import pymupdf
import re
from navec import Navec
from slovnet import NER

# Загрузка моделей (при первом запуске скачаются автоматически)
# Файлы: navec_news_v1.pkl и slovnet_ner_news_v1.tar
navec = Navec.load('navec_news_v1')
ner = NER.load('slovnet_ner_news_v1')

def clean_pdf_text(text: str) -> str:
    """Улучшенная чистка текста: склейка строк, удаление переносов, исправление пробелов"""
    # Убираем мягкие переносы (висячий дефис)
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # Склеиваем строки, не являющиеся концом предложения
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    text = ' '.join(merged)  # сплошной текст
    
    # Исправляем типичные OCR-ошибки с пунктуацией
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)   # пробелы перед знаками
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)  # один пробел после знака
    text = re.sub(r'(\w)\s*,\s*(\w)', r'\1, \2', text)
    text = re.sub(r'(\w)\s*\.\s*(\w)', r'\1. \2', text)
    
    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_text_from_pdf(pdf_path: str) -> str:
    """Извлекает текст из PDF и чистит его"""
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

def extract_names_with_slovnet(text: str, window_left: int = 60, window_right: int = 150):
    """
    Находит имена (PER) с помощью slovnet и возвращает список (имя, контекст)
    Контекст — это окно символов вокруг имени.
    """
    markup = ner(text)
    results = []
    
    for span in markup.spans:
        if span.type == 'PER':
            name = span.text.strip()
            # Фильтрация мусорных имён
            if len(name) < 3:
                continue
            if re.fullmatch(r'[А-Я]\.?(\s[А-Я]\.?)?', name):  # одиночные инициалы
                continue
            if re.search(r'\d', name):
                continue
            
            # Вырезаем контекстное окно (по символам)
            # У span есть start и stop – позиции в исходном тексте
            start = max(0, span.start - window_left)
            end = min(len(text), span.stop + window_right)
            context = text[start:end].strip()
            # Убираем начальный мусор (например, "...")
            context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
            results.append((name, context))
    
    # Дедупликация по имени + началу контекста
    unique = []
    seen = set()
    for name, ctx in results:
        key = (name, ctx[:100])
        if key not in seen:
            seen.add(key)
            unique.append((name, ctx))
    return unique

# Пример использования
if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    
    print("Извлечение текста из PDF...")
    full_text = get_text_from_pdf(pdf_path)
    
    print("Поиск имён людей (slovnet)...")
    names_contexts = extract_names_with_slovnet(full_text)
    
    print(f"\nНайдено уникальных упоминаний: {len(names_contexts)}\n")
    for name, ctx in names_contexts[:20]:  # выводим первые 20
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx) > 200 else ''}")
        print("-" * 70)

FileNotFoundError: [Errno 2] No such file or directory: 'navec_news_v1'

In [ ]:
import pymupdf
import re
from transformers import pipeline

# Загрузка модели NER (автоматически скачается при первом запуске)
ner_pipeline = pipeline("ner", model="Gherman/bert-base-NER-Russian", aggregation_strategy="simple")

def clean_pdf_text(text: str) -> str:
    """Чистка текста: склейка строк, удаление переносов, исправление пробелов"""
    # Убираем мягкие переносы
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # Склеиваем строки внутри предложений
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    text = ' '.join(merged)
    
    # Убираем лишние пробелы вокруг знаков
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

def extract_names_with_transformers(text: str, window_left: int = 60, window_right: int = 150):
    """Извлечение имён (PER) с помощью BERT, с контекстным окном"""
    # Получаем сущности из модели
    entities = ner_pipeline(text)
    
    results = []
    for ent in entities:
        if ent['entity_group'] == 'PER':  # только люди
            name = ent['word'].strip()
            # Очистка имени: бывает, что модель выдает подстроки, склеиваем если надо
            # Простая фильтрация
            if len(name) < 3:
                continue
            if re.fullmatch(r'[А-Я]\.?(\s[А-Я]\.?)?', name):
                continue
            # Контекстное окно (по символам)
            start = max(0, ent['start'] - window_left)
            end = min(len(text), ent['end'] + window_right)
            context = text[start:end].strip()
            context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
            results.append((name, context))
    
    # Дедупликация
    unique = []
    seen = set()
    for name, ctx in results:
        key = (name, ctx[:100])
        if key not in seen:
            seen.add(key)
            unique.append((name, ctx))
    return unique

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    
    print("Извлечение текста из PDF...")
    full_text = get_text_from_pdf(pdf_path)
    
    print("Поиск имён людей (BERT)...")
    names_contexts = extract_names_with_transformers(full_text)
    
    print(f"\nНайдено уникальных упоминаний: {len(names_contexts)}\n")
    for name, ctx in names_contexts[:20]:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx) > 200 else ''}")
        print("-" * 70)

c:\Users\nikol\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nikol\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikol\.cache\huggingface\hub\models--Gherman--bert-base-NER-Russian. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python 

Извлечение текста из PDF...
Поиск имён людей (BERT)...

Найдено уникальных упоминаний: 0



In [ ]:
import pymupdf
import re
from transformers import pipeline

ner_pipeline = pipeline("ner", model="Gherman/bert-base-NER-Russian", aggregation_strategy="simple")

def clean_pdf_text(text: str) -> str:
    """Минимальная чистка: только убираем дефисные переносы и лишние пробелы"""
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # Не склеиваем строки сильно – оставляем как есть, только заменяем несколько пробелов
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    
    # Отладочный вывод: первые 500 символов текста
    print("Текст (первые 500 символов):")
    print(full_text[:500])
    print("\n" + "="*80 + "\n")
    
    print("Поиск всех сущностей через BERT...")
    entities = ner_pipeline(full_text)
    
    if not entities:
        print("Модель не нашла НИ ОДНОЙ сущности. Возможно, текст не подходит для этой модели.")
    else:
        print(f"Найдено сущностей: {len(entities)}")
        for ent in entities[:30]:  # покажем первые 30
            print(f"Тип: {ent['entity_group']}, Текст: {ent['word']}, позиция: {ent['start']}-{ent['end']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7340.11it/s]


Текст (первые 500 символов):
ПРОЯКТАММ tc ix СТ»АЯ.СОВДНН«ИТ1СЫ ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС И РЕЖЕВСКОГО ГОРОДСКОГО СОВЕТА НАРОДНЫХ ДЕПУТАТОВ коммунизме ВТОРНИК, 8 ИЮЛЯ 1980 г. СМсТРЧ / ) Ь _ с ѵ П *«с 1 им. но 81 (6543) o S '4 L S o ° Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов. ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами. Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на гр


Поиск всех сущностей через BERT...
Найдено сущностей: 30
Тип: COUNTRY, Текст: Р, позиция: 41-42
Тип: COUNTRY, Текст: ##Е, позиция: 42-43
Тип: REGION, Текст: ##Ж, позиция: 43-44
Тип: DISTRICT, Текст: ##Е, позиция: 44-45
Тип: REGION, Текст: ##ВС, позиция: 45-47
Тип: COUNTRY, Текст: Р, позиция: 79-80
Тип: COUNTRY, Текст: ##Е, позиция: 80-81
Тип: REGION, Текст: ##Ж, позиция: 81-82
Тип: DISTRICT, Текст: ##Е, позиция: 82-83
Тип: FIRST_NAME, Текст: Сергей, позиция: 331-3

In [ ]:
import pymupdf
import re
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc

# ---------- Инициализация Natasha ----------
segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

# ---------- Ваша функция чистки (без изменений) ----------
def clean_pdf_text(text: str) -> str:
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    cleaned = '\n'.join(merged)
    cleaned = re.sub(r'\s+([.,!?;:])', r'\1', cleaned)
    cleaned = re.sub(r'([„"«(])\s+', r'\1', cleaned)
    cleaned = re.sub(r'\s+([»"»)]),', r'\1', cleaned)
    return cleaned

def get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

# ---------- Извлечение имён с контекстом (исправлено) ----------
def extract_names_with_context(text: str, window_left=50, window_right=150):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)
    
    results = []
    for span in doc.spans:
        if span.type != "PER":
            continue
        name = span.text.strip()
        # Фильтрация коротких/мусорных имён
        if len(name) < 3:
            continue
        # Вырезаем контекст по символьным позициям (span.start и span.stop работают!)
        start = max(0, span.start - window_left)
        end = min(len(text), span.stop + window_right)
        context = text[start:end].strip()
        # Убираем начальные обрывки (если есть)
        context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
        results.append((name, context))
    
    # Простая дедупликация (по точному совпадению имени и контекста)
    unique = []
    seen = set()
    for name, ctx in results:
        key = (name, ctx[:100])  # сравниваем начало контекста
        if key not in seen:
            seen.add(key)
            unique.append((name, ctx))
    return unique

# ---------- Запуск ----------
if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    
    print("Текст извлечён, ищем имена...\n")
    names_contexts = extract_names_with_context(full_text)
    
    print(f"Найдено уникальных упоминаний: {len(names_contexts)}\n")
    for name, ctx in names_contexts[:20]:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx) > 200 else ''}")
        print("-" * 70)

Текст извлечён, ищем имена...

Найдено уникальных упоминаний: 145

Имя: Сергей Алексеевич Горбу-
Контекст: работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов.
ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами.
Звено выполняет свой план на 110 про- равдяющий В. Д...
----------------------------------------------------------------------
Имя: Звено
Контекст: отая пора сенокоса определена строгими границами.
Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы. З...
----------------------------------------------------------------------
Имя: В. Д. Конев
Контекст: .
Звено выполняет свой план на 110 про- равдяющий В. Д. Конев), нов — на граблях, Михаил совхоза «Глинский» наби- Митаевич Митаев и Сергей рает темпы. Заготовкой кор- Михайлович Ермаков— сто- мов здес...
----------------------------------------------------------------------
Имя: Михаил
Контекст

In [ ]:
import pymupdf
import re

def clean_pdf_text(text: str) -> str:
    """Ваша исходная чистка (без изменений)"""
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    cleaned = '\n'.join(merged)
    cleaned = re.sub(r'\s+([.,!?;:])', r'\1', cleaned)
    cleaned = re.sub(r'([„"«(])\s+', r'\1', cleaned)
    cleaned = re.sub(r'\s+([»"»)]),', r'\1', cleaned)
    return cleaned

def get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return clean_pdf_text(text)

def extract_names_regex(text: str, window: int = 200):
    """
    Извлекает имена людей по шаблонам:
    - Фамилия Имя Отчество (с заглавных)
    - Фамилия И. О.
    - И. О. Фамилия
    """
    # Шаблон: сначала ищем полные имена (три слова с заглавной буквы)
    pattern1 = r'\b([А-Я][а-я]+)\s+([А-Я][а-я]+)\s+([А-Я][а-я]+)\b'
    # Шаблон: Фамилия И. О. (например Иванов И. И.)
    pattern2 = r'\b([А-Я][а-я]+)\s+([А-Я])\.\s*([А-Я])\.\b'
    # Шаблон: И. О. Фамилия (например И. И. Иванов)
    pattern3 = r'\b([А-Я])\.\s*([А-Я])\.\s+([А-Я][а-я]+)\b'
    # Объединяем все паттерны
    combined = re.compile(f'({pattern1})|({pattern2})|({pattern3})', re.UNICODE)
    
    results = []
    for match in combined.finditer(text):
        # Берём первую непустую группу
        name = match.group(0).strip()
        if len(name) < 5:
            continue
        # Контекст: окно вокруг совпадения
        start = max(0, match.start() - window)
        end = min(len(text), match.end() + window)
        context = text[start:end].strip()
        # Убираем обрезанные предложения в начале
        context = re.sub(r'^[^.!?]*?\.\.\.', '', context)
        results.append((name, context))
    
    # Дедупликация по имени (первые 100 символов контекста)
    unique = {}
    for name, ctx in results:
        key = (name, ctx[:100])
        if key not in unique:
            unique[key] = (name, ctx)
    return list(unique.values())

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    print("Поиск имён по регулярным выражениям...")
    names_contexts = extract_names_regex(full_text)
    print(f"Найдено уникальных упоминаний: {len(names_contexts)}\n")
    for name, ctx in names_contexts[:30]:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx) > 200 else ''}")
        print("-" * 70)

Поиск имён по регулярным выражениям...
Найдено уникальных упоминаний: 35

Имя: Сергей Алексеевич Горбу
Контекст: унизме ВТОРНИК, 8 ИЮЛЯ 1980 г.
СМсТРЧ / ) Ь _ с ѵ П *«с 1 им.
но 81 (6543) o S '4 L S o ° Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Ара- работает на подборщике, маншовском отделении (уп- ...
----------------------------------------------------------------------
Имя: В. Д. Конев
Контекст: ра- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов.
ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами.
Звено выполняет свой план на 110 про- равдяющий ...
----------------------------------------------------------------------
Имя: Анатолий Васильевич Ежов
Контекст: боту и воскресенье трудно было кого-то увидеть на улицах Черемисского и во всех отделениях совхоза: все вышли на заготовку кормов.
Лучше всех идет работа на сенокосах в нервом отделении, где трудится ...
------------------------------------------------------------------

In [ ]:
import pymupdf
import re
import spacy

# Загрузка русской модели spaCy
nlp = spacy.load("ru_core_news_sm")

def master_clean_text(text: str) -> str:
    """
    Убирает всё, что мешает NER:
    - склеивает строки
    - удаляет мягкие переносы
    - убирает точки внутри слов (например, ком.мунистического → коммунистического)
    - исправляет дефисные переносы
    - удаляет лишние пробелы
    """
    # 1. Склеиваем висячие дефисы и мягкие переносы (включая с пробелом и переводом строки)
    text = re.sub(r'(\w+)­\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # 2. Убираем точки, которые явно внутри слова (например, "ком.мунистического" -> "коммунистического")
    #    Оставляем точки после инициалов (А. А. Баранова) – их не трогаем.
    #    Сначала заменяем точку между буквами, если после точки идёт буква.
    text = re.sub(r'(?<=[а-яА-Я])\.(?=[а-яА-Я])', '', text)
    
    # 3. Удаляем символы «», " и т.п., если они мешают, но оставляем кавычки для имён (не критично)
    text = re.sub(r'[«»“”]', '"', text)
    
    # 4. Склеиваем строки, которые не заканчиваются точкой или знаком восклицания/вопроса
    lines = text.split('\n')
    merged = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if merged and not merged[-1].endswith(('.', '!', '?')):
            merged[-1] += ' ' + line
        else:
            merged.append(line)
    text = ' '.join(merged)
    
    # 5. Убираем лишние пробелы вокруг знаков препинания
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.,!?;:])\s+', r'\1 ', text)
    
    # 6. Убираем повторяющиеся пробелы
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw = "\n".join(page.get_text() for page in doc)
    doc.close()
    return master_clean_text(raw)

def extract_names_spacy_with_fallback(text: str, window_left=60, window_right=150):
    """
    Сначала пытается найти имена через spaCy.
    Если результат пустой или содержит мусор, использует regex для поиска ФИО.
    """
    doc = nlp(text)
    results = []
    
    # Через spaCy
    for ent in doc.ents:
        if ent.label_ == "PER":
            name = ent.text.strip()
            # Фильтрация коротких и неинформативных
            if len(name) < 3:
                continue
            # Пропускаем одиночные инициалы
            if re.fullmatch(r'[А-Я]\.(\s[А-Я]\.)?', name):
                continue
            # Контекст – окно вокруг имени
            start = max(0, ent.start_char - window_left)
            end = min(len(text), ent.end_char + window_right)
            ctx = text[start:end].strip()
            results.append((name, ctx))
    
    # Если мало результатов (менее 2), добавим поиск по регуляркам
    if len(results) < 2:
        # Шаблоны: Фамилия Имя Отчество, Фамилия И.О., И.О. Фамилия
        patterns = [
            r'\b([А-Я][а-я]+)\s+([А-Я][а-я]+)\s+([А-Я][а-я]+)\b',  # Иванов Иван Иванович
            r'\b([А-Я][а-я]+)\s+([А-Я])\.\s*([А-Я])\.\b',          # Иванов И. И.
            r'\b([А-Я])\.\s*([А-Я])\.\s+([А-Я][а-я]+)\b',          # И. И. Иванов
        ]
        combined = re.compile('|'.join(patterns), re.UNICODE)
        for match in combined.finditer(text):
            name = match.group(0).strip()
            if len(name) < 5:
                continue
            start = max(0, match.start() - window_left)
            end = min(len(text), match.end() + window_right)
            ctx = text[start:end].strip()
            results.append((name, ctx))
    
    # Дедупликация
    unique = {}
    for name, ctx in results:
        # нормализуем имя: убираем точки в конце (если есть)
        name = re.sub(r'\.$', '', name)
        key = (name, ctx[:100])
        if key not in unique:
            unique[key] = (name, ctx)
    return list(unique.values())

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"
    full_text = get_text_from_pdf(pdf_path)
    
    print("Поиск имён (spaCy + fallback)...")
    persons = extract_names_spacy_with_fallback(full_text)
    
    print(f"Найдено уникальных: {len(persons)}\n")
    for name, ctx in persons[:30]:
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:200]}{'...' if len(ctx)>200 else ''}")
        print("-" * 70)

Поиск имён (spaCy + fallback)...
Найдено уникальных: 137

Имя: Т Р А Д Н Ы Е П
Контекст: / ) Ь _ с ѵ П *"с 1 им. но 81 (6543) o S '4 L S o ° Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов. ЗЕЛЕН...
----------------------------------------------------------------------
Имя: Сергей Алексеевич Горбу-
Контекст: да в Ара- работает на подборщике, маншовском отделении (уп- Сергей Алексеевич Горбу- центов. ЗЕЛЕНАЯ ЖАТВА лотая пора сенокоса определена строгими границами. Звено выполняет свой план на 110 проравдяю...
----------------------------------------------------------------------
Имя: В. Д. Конев
Контекст: ми границами. Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на граблях, Михаил совхоза "Глинский" наби- Митаевич Митаев и Сергей рает темпы. Заготовкой кор- Михайлович Ермаков— сто...
----------------------------------------------------------------------
Имя: Михаил
Контекс

In [ ]:
# Improved Russian NER pipeline for Soviet newspapers (standalone example)
# This code demonstrates a small, self-contained NER pipeline using
# classic NLP techniques suitable for historic, noisy texts.
# It does NOT rely on network access or heavy models.

import re
from typing import List, Tuple

# 1) Basic tokenizer tuned for Cyrillic and newspapers-style text
TOKEN_RE = re.compile(r"[\w-]+|[.,!?;:"')()]|[А-Яа-яЁё]+", re.UNICODE)

# 2) Lightweight gazetteer tailored for Soviet-era entities
GAZETTEER = {
    # Persons (often with patronymics in newspapers)
    'Ленин': 'PERSON',
    'Сталин': 'PERSON',
    'Крупская': 'PERSON',
    'Молотов': 'PERSON',
    # Places
    'Москва': 'LOCATION',
    'Сталинград': 'LOCATION',  # historic name
    'Днепр': 'LOCATION',
    # Organizations
    'Совет': 'ORG',
    'Коммунистическая': 'ORG',
    'Правда': 'ORG',
    'Известия': 'ORG',
}

# 3) Simple suffix-based NER cues (dates, numbers, roles)
DATE_RE = re.compile(r"\b(19\d{2}|20\d{2})\b")
ROLE_RE = re.compile(r"\b(генерал|министр|председатель|сообщение|социалист|кандидат|рабочий)\b", re.IGNORECASE)

# 4) Heuristic tagging function

def ner_pipeline(text: str) -> List[Tuple[str, str]]:
    tokens = [m.group(0) for m in re.finditer(r"\w+|[^\s]", text, re.UNICODE)]
    results: List[Tuple[str, str]] = []

    for t in tokens:
        tag = 'O'
        # Gazetteer lookup (case-sensitive to preserve era spellings; adjust as needed)
        if t in GAZETTEER:
            tag = GAZETTEER[t]
        # Dates
        if DATE_RE.fullmatch(t) or DATE_RE.search(text):
            if DATE_RE.search(f" {t} "):
                tag = 'DATE'
        # Roles
        if ROLE_RE.search(t):
            tag = 'ROLE'
        # Fallback: ALLCAP tokens might be NER-like (e.g., Soviet-era acronyms)
        if t.isupper() and len(t) > 1:
            tag = 'ORG'
        if tag != 'O':
            results.append((t, tag))
    return results

# 5) Demonstration with a sample Soviet-era newspaper excerpt
sample = (
    "Сегодня in Москва проходит очередное заседание Совета. Председатель Никита Хрущёв обратив внимание на экономику, "
    "вышел доклад Молотова. Газета Правда сообщает о росте промышленного производства за 1955 год."
)

# Run the pipeline and print results
for token, tag in ner_pipeline(sample):
    print(f"{token:>12} -> {tag}")


SyntaxError: unterminated string literal (detected at line 10) (3522277130.py, line 10)

In [ ]:
import os
import re
import pymupdf  # pip install pymupdf
import spacy

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Установи заранее ru_core_news_lg:
# pip install https://github.com/explosion/spacy-models/releases/download/ru_core_news_lg-3.7.0/ru_core_news_lg-3.7.0-py3-none-any.whl
nlp = spacy.load("ru_core_news_lg")


# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    """
    Эвристика: строка считается "шумом", если в ней больше
    threshold доли небуквенно-цифровых символов.
    Используется, чтобы отсечь артефакты OCR, шапки и пр.
    """
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    """
    Исправляет переносы слов внутри одной строки:
    'Ерма-ков' -> 'Ермаков'
    (аккуратно: только кириллица)
    """
    line = re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )
    return line


def merge_lines_gazette_style(lines):
    """
    Склеивает строки газетной полосы в более осмысленные абзацы.
    Эвристики:
    - пропускаем "шумные" строки;
    - не склеиваем, если строка похожа на заголовок (ALL CAPS, короткая);
    - склеиваем, если предыдущая строка не заканчивается на .!? и т.п.
    """
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 60:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.7  # почти все буквы заглавные

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            # пустая строка -> новый абзац
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            # выбрасываем мусорные строки
            continue

        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        # новая строка похожа на заголовок — начинаем новый абзац
        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        # если предыдущая строка не заканчивается знаком конца предложения — склеиваем
        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    """
    Усиленный препроцессинг под советские газеты:
    - удаляем "мягкие" переносы и склеиваем переносы через дефис (между строками);
    - приводим кавычки к одному виду;
    - убираем лишний мусор;
    - аккуратно склеиваем строки в абзацы.
    """

    # 1. Убираем висячие дефисы и мягкие переносы МЕЖДУ строками
    # слово­\nслово или слово-\nслово
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)

    # 2. Приводим кавычки к одному виду
    text = re.sub(r"[«»“”]", '"', text)

    # 3. Делим по строкам
    lines = text.split("\n")

    # 4. Чистим отдельные строки
    cleaned_lines = []
    for line in lines:
        l = line.rstrip()
        if is_mostly_noise(l):
            continue
        cleaned_lines.append(l)

    # 5. Склейка строк с учётом газетных колонок
    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)

    # 6. Финальная нормализация пробелов и пунктуации
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    # восстановим абзацные разрывы:
    text = re.sub(r"\s*\n\s*", "\n", text)

    return text.strip()


# ============================================
# ЧТЕНИЕ ТЕКСТА ИЗ PDF
# ============================================

def get_text_from_pdf(pdf_path: str) -> str:
    """
    Извлекает текст из PDF и прогоняет через master_clean_text.
    """
    doc = pymupdf.open(pdf_path)
    raw_pages = []
    for page in doc:
        # можно поэкспериментировать с режимами:
        # raw_pages.append(page.get_text("blocks"))
        raw_pages.append(page.get_text())
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)


# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН: SPACY + REGEX + ПОДПИСИ
# ============================================

# Базовый список стоп-слов, которые часто выглядят как фамилии,
# но в газетном контексте почти всегда не люди (можно расширять).
NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "Реж", "Режевского",
    "Совет", "Совета", "КПСС", "СССР",
    "Глинский", "Глинская", "Глинское",
}


def extract_persons(text: str):
    """
    Главная функция.
    Возвращает список (name, context, method), где:
    - name: строка с именем,
    - context: 1-2 предложения контекста,
    - method: 'spacy', 'regex_*', 'signature'.
    Максимизируем recall, фильтрация минимальная.
    """

    results = []

    # 1. Разбиваем текст на абзацы
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- 1) NER через spaCy по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            sent = ent.sent
            # контекст: предложение + следующее
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append((name, ctx_text, "spacy"))

    # --- 2) REGEX по всему тексту ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    # 2.1. Полные ФИО
    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    # 2.2. Фамилия + инициалы
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    # 2.3. Инициалы + фамилия
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    # 2.4. Одна инициала + фамилия
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label):
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue

            # лёгкая фильтрация по стоп-словам
            skip = False
            for token in NON_PERSON_TOKENS:
                if token in name:
                    skip = True
                    break
            if skip:
                continue

            start, end = m.span()
            # находим предложение для контекста
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break
            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()

            results.append((name, ctx_text, method_label))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- 3) Подписи в конце заметок ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append((short_name, ctx_text, "signature"))

    # --- 4) Дедупликация (минимальная) ---
    unique = {}
    for name, ctx, method in results:
        clean_name = re.sub(r"\.$", "", name).strip()
        key = (clean_name, ctx[:120])
        if key not in unique:
            unique[key] = (clean_name, ctx, method)

    return list(unique.values())


# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"

    full_text = get_text_from_pdf(pdf_path)

    print("Поиск имён (spaCy + расширенный regex + подписи)...")
    persons = extract_persons(full_text)

    print(f"Найдено уникальных (после лёгкой дедупликации): {len(persons)}\n")
    for name, ctx, method in persons[:50]:
        print(f"Источник: {method}")
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:300]}{'...' if len(ctx) > 300 else ''}")
        print("-" * 80)

Поиск имён (spaCy + расширенный regex + подписи)...
Найдено уникальных (после лёгкой дедупликации): 163

Источник: spacy
Имя: Алексеевич Горбуцентов
Контекст: ПРОЯКТАММ tc ix СТ"АЯ.СОВДНН"ИТ1СЫ ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС И РЕЖЕВСКОГО ГОРОДСКОГО СОВЕТА НАРОДНЫХ ДЕПУТАТОВ коммунизме ВТОРНИК, 8 ИЮЛЯ 1980 г. СМсТРЧ / ) Ь с ѵ П *"с 1 им. но 81 (6543) o S '4 L S o Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Араработает на подборщике, м...
--------------------------------------------------------------------------------
Источник: spacy
Имя: В. Д. Конев
Контекст: Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на граблях, Михаил совхоза "Глинский" набиМитаевич Митаев и Сергей рает темпы. Заготовкой корМихайлович Ермаков— стомов здесь занимается комгометамй, Иван Ёвдокимоплексное звено под рукович Дудкин и Анатолий водством Николая ПавлоДмит...
--------------------------------------------------------------------------------
Источник: spacy
Имя: Михаил


In [ ]:
# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН: SPACY + REGEX + ПОДПИСИ (улучшенная фильтрация)
# ============================================

# Стоп-слова, которые почти всегда не люди
NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "Реж", "Режевского",
    "Совет", "Совета", "КПСС", "СССР",
    "Глинский", "Глинская", "Глинское",
    "Советов", "Время", "Боевой", "Комм", "Брак",
}

# Стоп-слова в нижнем регистре
NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}


def looks_like_garbage_name(name: str) -> bool:
    """
    Эвристика для отбрасывания откровенного мусора.
    Мы НЕ пытаемся быть слишком строгими, чтобы не убить recall.
    """
    text = name.strip()

    # Если начинается со строчной кириллицы -> скорее всего не имя
    if re.match(r"^[а-яё]", text):
        return True

    # Одно слово, очень длинное и без пробелов -> похоже на склейку OCR
    if " " not in text and len(text) > 18:
        return True

    # Содержит явный мусор: #, |, /, скобки и т.п.
    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    # Содержит подряд гласных/согласных очень странно? (упрощённо)
    # Отбрасываем, если почти все буквы - гласные или почти все - согласные
    # (но только для одного слова, чтобы не затронуть нормальные ФИО)
    parts = text.split()
    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    # если в имени есть подряд кусок типа "корМихайлович" -> проверим каждое слово
    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def extract_persons(text: str):
    """
    Возвращает список (name, context, method), где:
    - name: имя,
    - context: 1–2 предложения,
    - method: 'spacy', 'regex_*', 'signature'.
    """

    results = []

    # 1. Абзацы
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- 1) spaCy по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            # грубый отбор мусора
            if looks_like_garbage_name(name):
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append((name, ctx_text, "spacy"))

    # --- 2) REGEX по всему тексту ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label):
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()

            results.append((name, ctx_text, method_label))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- 3) Подписи в конце заметок ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append((short_name, ctx_text, "signature"))

    # --- 4) Дедупликация ---
    unique = {}
    for name, ctx, method in results:
        clean_name = re.sub(r"\.$", "", name).strip()
        key = (clean_name, ctx[:120])
        if key not in unique:
            unique[key] = (clean_name, ctx, method)

    return list(unique.values())

In [ ]:
import os
import re
import pymupdf  # pip install pymupdf
import spacy

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")


# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    """
    Эвристика: строка считается "шумом", если в ней больше
    threshold доли небуквенно-цифровых символов.
    Используется, чтобы отсечь артефакты OCR, шапки и пр.
    """
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    """
    Исправляет переносы слов внутри одной строки:
    'Ерма-ков' -> 'Ермаков' (только кириллица)
    """
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def merge_lines_gazette_style(lines):
    """
    Склеивает строки газетной полосы в абзацы.
    Эвристики:
    - пропускаем "шумные" строки;
    - не склеиваем, если строка похожа на заголовок (ALL CAPS, короткая);
    - склеиваем, если предыдущая строка не заканчивается на .!? и т.п.
    """
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 60:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.7  # почти все буквы заглавные

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    """
    Усиленный препроцессинг под советские газеты.
    """

    # 1. Убираем висячие дефисы и мягкие переносы МЕЖДУ строками
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)

    # 2. Нормализуем кавычки
    text = re.sub(r"[«»“”]", '"', text)

    # 3. Построчная обработка
    lines = text.split("\n")

    cleaned_lines = []
    for line in lines:
        l = line.rstrip()
        if is_mostly_noise(l):
            continue
        cleaned_lines.append(l)

    # 4. Склейка строк
    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)

    # 5. Финальная чистка
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    return text.strip()


# ============================================
# ЧТЕНИЕ ТЕКСТА ИЗ PDF
# ============================================

def get_text_from_pdf(pdf_path: str) -> str:
    """
    Извлекает текст из PDF и прогоняет через master_clean_text.
    """
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)


# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН: SPACY + REGEX + ПОДПИСИ (улучшенная фильтрация)
# ============================================

# Стоп-слова, которые почти всегда не люди
NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "Реж", "Режевского",
    "Совет", "Совета", "КПСС", "СССР",
    "Глинский", "Глинская", "Глинское",
    "Советов", "Время", "Боевой", "Комм", "Брак",
}

# Стоп-слова в нижнем регистре
NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}


def looks_like_garbage_name(name: str) -> bool:
    """
    Эвристика для отбрасывания откровенного мусора.
    Мы НЕ пытаемся быть слишком строгими, чтобы не убить recall.
    """
    text = name.strip()

    # Если начинается со строчной кириллицы -> скорее всего не имя
    if re.match(r"^[а-яё]", text):
        return True

    # Одно слово, очень длинное и без пробелов -> похоже на склейку OCR
    if " " not in text and len(text) > 18:
        return True

    # Содержит явный мусор: #, |, /, скобки и т.п.
    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    # Слишком странное одно слово: только гласные или только согласные
    parts = text.split()
    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    # Слова из нашего нижнего списка
    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def extract_persons(text: str):
    """
    Возвращает список (name, context, method), где:
    - name: имя,
    - context: 1–2 предложения,
    - method: 'spacy', 'regex_*', 'signature'.
    """

    results = []

    # 1. Абзацы
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- 1) spaCy по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if looks_like_garbage_name(name):
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append((name, ctx_text, "spacy"))

    # --- 2) REGEX по всему тексту ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label):
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()

            results.append((name, ctx_text, method_label))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- 3) Подписи в конце заметок ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append((short_name, ctx_text, "signature"))

    # --- 4) Дедупликация ---
    unique = {}
    for name, ctx, method in results:
        clean_name = re.sub(r"\.$", "", name).strip()
        key = (clean_name, ctx[:120])
        if key not in unique:
            unique[key] = (clean_name, ctx, method)

    return list(unique.values())


# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\magazines\1980-07-08_Правда коммунизма 1980 081.pdf"

    full_text = get_text_from_pdf(pdf_path)

    print("Поиск имён (ru_core_news_lg + regex + подписи, с фильтрацией мусора)...")
    persons = extract_persons(full_text)

    print(f"Найдено уникальных (после лёгкой дедупликации): {len(persons)}\n")
    for name, ctx, method in persons[:50]:
        print(f"Источник: {method}")
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:300]}{'...' if len(ctx) > 300 else ''}")
        print("-" * 80)

Поиск имён (ru_core_news_lg + regex + подписи, с фильтрацией мусора)...
Найдено уникальных (после лёгкой дедупликации): 144

Источник: spacy
Имя: Алексеевич Горбуцентов
Контекст: ПРОЯКТАММ tc ix СТ"АЯ.СОВДНН"ИТ1СЫ ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС И РЕЖЕВСКОГО ГОРОДСКОГО СОВЕТА НАРОДНЫХ ДЕПУТАТОВ коммунизме ВТОРНИК, 8 ИЮЛЯ 1980 г. СМсТРЧ / ) Ь с ѵ П *"с 1 им. но 81 (6543) o S '4 L S o Т У Т С Т Р А Д Н Ы Е П РО Ц ЕН ТЫ Сенокосная страда в Араработает на подборщике, м...
--------------------------------------------------------------------------------
Источник: spacy
Имя: В. Д. Конев
Контекст: Звено выполняет свой план на 110 проравдяющий В. Д. Конев), нов — на граблях, Михаил совхоза "Глинский" набиМитаевич Митаев и Сергей рает темпы. Заготовкой корМихайлович Ермаков— стомов здесь занимается комгометамй, Иван Ёвдокимоплексное звено под рукович Дудкин и Анатолий водством Николая ПавлоДмит...
--------------------------------------------------------------------------------
Источник

In [ ]:
import os
import re
import pymupdf  # pip install pymupdf
import spacy

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")


# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА ПОД ГАЗЕТЫ
# ============================================

# Регулярки для шапок/колонтитулов «Правды коммунизма»
HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)


def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    """
    Эвристика: строка считается "шумом", если в ней больше
    threshold доли небуквенно-цифровых символов.
    Используется, чтобы отсечь артефакты OCR, шапки и пр.
    """
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    """
    Исправляет переносы слов внутри одной строки:
    'Ерма-ков' -> 'Ермаков' (только кириллица)
    """
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    """
    Склеивает разреженные капс-надписи:
    'Г О Л Е Н Д У Х И Н' -> 'ГОЛЕНДУХИН'
    Работает только для последовательностей из >= 3 заглавных кириллических букв.
    """
    def repl(match):
        return match.group(0).replace(" ", "")

    # последовательности вида: Г О Л Е Н Д У Х И Н
    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines):
    """
    Склеивает строки газетной полосы в более осмысленные абзацы.
    Эвристики:
    - пропускаем "шумные" строки;
    - не склеиваем, если строка похожа на заголовок (ALL CAPS, короткая);
    - склеиваем, если предыдущая строка не заканчивается на .!? и т.п.
    """
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        # почти все буквы заглавные
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            # пустая строка -> новый абзац
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        # исправляем разреженные капсы и переносы внутри строки
        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        # новая строка похожа на заголовок — начинаем новый абзац
        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        # если предыдущая строка не заканчивается знаком конца предложения — склеиваем
        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    """
    Усиленный препроцессинг под советские газеты:
    - убираем висячие дефисы и мягкие переносы МЕЖДУ строками;
    - удаляем шапки/колонтитулы, TV-программу и явные служебные блоки;
    - исправляем разреженные капсы;
    - аккуратно склеиваем строки в абзацы;
    - чистим пунктуацию и пробелы.
    """

    # 1. Убираем висячие дефисы и мягкие переносы МЕЖДУ строками
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)

    # 2. Нормализуем кавычки
    text = re.sub(r"[«»“”]", '"', text)

    # 3. Построчная обработка
    lines = text.split("\n")

    cleaned_lines = []
    for line in lines:
        l = line.rstrip()

        # шапки/футеры/тв-программа
        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        # явные служебные вещи (индексы, телефоны, адреса редакции)
        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    # 4. Склейка строк
    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)

    # 5. Финальная чистка
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    return text.strip()


# ============================================
# ЧТЕНИЕ ТЕКСТА ИЗ PDF
# ============================================

def get_text_from_pdf(pdf_path: str) -> str:
    """
    Извлекает текст из PDF и прогоняет через master_clean_text.
    """
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)


# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН: SPACY + REGEX + ПОДПИСИ
# ============================================

# Базовый список стоп-слов, которые часто выглядят как фамилии,
# но в газетном контексте почти всегда не люди (можно расширять).
NON_PERSON_TOKENS = {
    # шапки, партии, органы
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",

    # география / хозструктуры, часто как фамилии
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",

    # газета, рубрики
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",

    # разные существительные, часто ошибочно как PER
    "Советов", "Время", "Боевой", "Комм", "Брак",
}

# Стоп-слова в нижнем регистре
NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}


def looks_like_garbage_name(name: str) -> bool:
    """
    Эвристика для отбрасывания откровенного мусора.
    Стараемся не убать recall, но отсечь явно сломанные сущности.
    """
    text = name.strip()

    # Если начинается со строчной кириллицы -> скорее всего не имя
    if re.match(r"^[а-яё]", text):
        return True

    # Одно слово, очень длинное и без пробелов -> похоже на склейку OCR
    if " " not in text and len(text) > 18:
        return True

    # Содержит очевидный мусор: #, |, /, скобки и т.п.
    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    # Одно слово: только гласные или только согласные
    parts = text.split()
    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    # Подозрительные подстроки, типичные для разъехавшихся слов ("городского", "областного" и т.п.)
    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        # типа "РДСК", "ГОРДСК" и т.п.
        return True

    # Слова из нашего нижнего списка
    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def extract_persons(text: str):
    """
    Возвращает список (name, context, method), где:
    - name: имя,
    - context: 1–2 предложения,
    - method: 'spacy', 'regex_*', 'signature'.
    """

    results = []

    # 1. Абзацы
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- 1) spaCy по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if looks_like_garbage_name(name):
                continue
            # Если содержит явно стоп-слова из шапок/рубрик — отбрасываем
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append((name, ctx_text, "spacy"))

    # --- 2) REGEX по всему тексту ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label):
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()

            results.append((name, ctx_text, method_label))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- 3) Подписи в конце заметок ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append((short_name, ctx_text, "signature"))

    # --- 4) Дедупликация ---
    unique = {}
    for name, ctx, method in results:
        clean_name = re.sub(r"\.$", "", name).strip()
        key = (clean_name, ctx[:120])
        if key not in unique:
            unique[key] = (clean_name, ctx, method)

    return list(unique.values())


# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    pdf_path = r"C:\Users\nikol\Desktop\disser\newspapers\1990-01-01_Правда коммунизма 1990 001.pdf"

    full_text = get_text_from_pdf(pdf_path)

    print("Поиск имён (ru_core_news_lg + regex + подписи, с улучшенным препроцессингом)...")
    persons = extract_persons(full_text)

    print(f"Найдено уникальных (после лёгкой дедупликации): {len(persons)}\n")
    for name, ctx, method in persons[:50]:
        print(f"Источник: {method}")
        print(f"Имя: {name}")
        print(f"Контекст: {ctx[:300]}{'...' if len(ctx) > 300 else ''}")
        print("-" * 80)

Поиск имён (ru_core_news_lg + regex + подписи, с улучшенным препроцессингом)...
Найдено уникальных (после лёгкой дедупликации): 93

Источник: spacy
Имя: Михаил Вандышев
Контекст: Среди женщмн-олераторов машинного доения на районных соревнованиях он, дояр из Клевакино, Михаил Вандышев замечательно смотрится. М ежду прочим.
--------------------------------------------------------------------------------
Источник: spacy
Имя: А. Шангина
Контекст: Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ?
--------------------------------------------------------------------------------
Источник: spacy
Имя: Мой
Контекст: На Мой взгляд долго мы будем чувствовать свою отчужденность, пока не станем, л подлинном смысле, хозяевами предприятий. Не скаж у} что за эти годы у нас ничего не изменилось.
--------------------------------------------------------------------------------
Источник: spacy
Имя: Съездов
Контекст: И ко ­ нечно, ж ду новых Съездов народных депутатов. Как и вся страна— с надеж дой, ПЕТР СЕРГЕЕВИЧКИЯШ К

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import fitz as pymupdf  # pip install pymupdf
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА ПОД ГАЗЕТЫ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})

# ============================================
# OCR НОРМАЛИЗАЦИЯ
# ============================================

def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))

# ============================================
# ПРЕПРОЦЕССИНГ ГАЗЕТ
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    # OCR нормализация уже над чистым текстом
    text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)

# ============================================
# ФИЛЬТРАЦИЯ ИМЁН — ЭВРИСТИКИ
# ============================================

def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()

    parts = text.split()
    # точно плохие одиночные токены
    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    # одно короткое слово: "Мой", "Ваши" и т.п.
    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 6 and token[0].isupper() and token[1:].islower():
            return True

    # одно длинное слово без суффикса фамилии — похожее на мусор OCR
    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        if len(only_cyr) >= 7:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                return True

    # подозрительные сочетания букв
    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    # старая логика
    if re.match(r"^[а-яё]", text):
        return True

    if " " not in text and len(text) > 18:
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True

# ============================================
# ДОOБУЧАЕМЫЙ ФИЛЬТР ИМЁН (SKLEARN)
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        feat = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),
            "ctx_has_pozicion": int(bool(re.search(r"директор|секретарь|председатель|первый секретарь|депутат", ctx, re.I))),
            "ctx_has_prof": int(bool(re.search(r"доярк|плавильщик|слесарь|редактор|журналист|учитель|врач", ctx, re.I))),
        }

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=200, class_weight="balanced")

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            # если модель не обучена — пускаем всё
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        # вероятность класса 1 (real person)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()

# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(
    text: str,
    use_trained_filter: bool = True,
    proba_threshold: float = 0.5
) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            # контекстные стоп‑фразы
            if bad_ctx_re.search(ent.sent.text):
                continue

            sent = ent.sent
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            # контекстные стоп‑фразы
            if bad_ctx_re.search(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» в конце абзаца ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered = [
            (cand.name, cand.ctx, cand.method)
            for cand, p in zip(final_candidates, proba)
            if p >= proba_threshold
        ]
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered

# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ JSON
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    """
    Ожидаемый формат JSON:
    [
      {
        "name": "А. Шангина",
        "ctx": "Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ? ...",
        "method": "spacy",
        "label": 1
      },
      ...
    ]
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)

# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 1. Путь к твоему размеченному JSON
    labeled_json = r"C:\path\to\labeled_names.json"  # <- поменяй на реальный путь

    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)
    else:
        print(f"Внимание: файл разметки не найден: {labeled_json}")
        print("Фильтр имён будет работать только на эвристиках (без ML).")

    # 2. Обработка папки с pdf-газетами
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)

Внимание: файл разметки не найден: C:\path\to\labeled_names.json
Фильтр имён будет работать только на эвристиках (без ML).

=== Обработка файла: 1990-01-01_Правда коммунизма 1990 001.pdf ===
Найдено уникальных имён: 51
[spacy] Михаил Вандышев :: Среди женщмн-олераторов машинного доения на районных соревнованиях он, дояр из Клевакино, Михаил Вандышев замечательно смотрится. М ежду прочим.
------------------------------------------------------------
[spacy] А. Шангина :: Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ?
------------------------------------------------------------
[spacy] Любови Михайловны Тоіпоркозой :: Неужели в веірху сидящ ие не понимают, что и сами могут оказаться в глазном отделении, оказаться в таких умелых и добрых руках, как у Любови Михайловны Тоіпоркозой. Скрро 'начнутся...
------------------------------------------------------------
[spacy] Кузьма Тимофеевич Швецов :: Мой собеседник Кузьма Тимофеевич Швецов родился в Ощепково, когда взошла заря Красного Октября. Именно та

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import fitz as pymupdf  # from PyMuPDF
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА ПОД ГАЗЕТЫ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
    "ГАЗЕТА", "Москва", "Москбьі", "Виновников"
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
    "Своем Министерстве", "Министерстве"
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})

# ============================================
# OCR НОРМАЛИЗАЦИЯ
# ============================================

def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))

# ============================================
# ПРЕПРОЦЕССИНГ ГАЗЕТ
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)

# ============================================
# ФИЛЬТРАЦИЯ ИМЁН — ЭВРИСТИКИ
# ============================================

def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()
    parts = text.split()

    # Точно плохие одиночные токены
    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    # Одно короткое слово: "Мой", "Ваши" и т.п.
    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 6 and token[0].isupper() and token[1:].islower():
            return True

    # Одно длинное слово без суффикса фамилии — похожее на мусор OCR
    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        if len(only_cyr) >= 7:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                return True

    # Подозрительные сочетания букв
    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    # Старая логика:
    if re.match(r"^[а-яё]", text):
        return True

    if " " not in text and len(text) > 18:
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True

def context_looks_like_index_list(ctx: str) -> bool:
    """
    Отфильтровывает куски, похожие на списки номеров/инициалов
    (много цифр/№/тире, мало букв).
    """
    letters = sum(ch.isalpha() for ch in ctx)
    digits = sum(ch.isdigit() for ch in ctx)
    specials = ctx.count("№") + ctx.count("N") + ctx.count("—") + ctx.count("-")
    length = len(ctx)

    if length == 0:
        return True

    if letters < 10 and (digits + specials) > 10:
        return True

    return False

def normalize_name_for_key(name: str) -> str:
    """
    Нормализация имени для дедупликации (грубо: убираем лишние пробелы).
    """
    n = re.sub(r"\.\s+", ". ", name)
    n = re.sub(r"\s{2,}", " ", n)
    n = n.strip()
    return n

# ============================================
# ДОOБУЧАЕМЫЙ ФИЛЬТР ИМЁН (SKLEARN)
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        feat = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),
            "ctx_has_pozicion": int(bool(re.search(r"директор|секретарь|председатель|первый секретарь|депутат", ctx, re.I))),
            "ctx_has_prof": int(bool(re.search(r"доярк|плавильщик|слесарь|редактор|журналист|учитель|врач|агроном", ctx, re.I))),
        }

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=200, class_weight="balanced")

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()

# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(
    text: str,
    use_trained_filter: bool = True,
    proba_threshold: float = 0.5
) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            if bad_ctx_re.search(sent.text):
                continue

            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)

            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            if bad_ctx_re.search(ctx_text):
                continue
            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» в конце абзаца ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            if context_looks_like_index_list(ctx_text):
                continue
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        clean_name = normalize_name_for_key(clean_name)
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered = [
            (cand.name, cand.ctx, cand.method)
            for cand, p in zip(final_candidates, proba)
            if p >= proba_threshold
        ]
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered

# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ JSON
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    """
    Ожидаемый формат JSON:
    [
      {
        "name": "А. Шангина",
        "ctx": "Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ? ...",
        "method": "spacy",
        "label": 1
      },
      ...
    ]
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)

# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 1. Путь к твоему размеченному JSON
    labeled_json = r"C:\Users\nikol\Desktop\disser\labeled_names.json"  # поменяй при необходимости

    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)
    else:
        print(f"Внимание: файл разметки не найден: {labeled_json}")
        print("Фильтр имён будет работать только на эвристиках (без ML).")

    # 2. Обработка папки с pdf-газетами
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)

=== NameFilterModel report ===
              precision    recall  f1-score   support

           0      1.000     0.800     0.889         5
           1      0.889     1.000     0.941         8

    accuracy                          0.923        13
   macro avg      0.944     0.900     0.915        13
weighted avg      0.932     0.923     0.921        13


=== Обработка файла: 1990-01-01_Правда коммунизма 1990 001.pdf ===
Ошибка чтения PDF: module 'fitz' has no attribute 'open'

=== Обработка файла: 1990-01-04_Правда коммунизма 1990 002.pdf ===
Ошибка чтения PDF: module 'fitz' has no attribute 'open'

=== Обработка файла: 1990-01-06_Правда коммунизма 1990 003.pdf ===
Ошибка чтения PDF: module 'fitz' has no attribute 'open'

=== Обработка файла: 1990-01-09_Правда коммунизма 1990 004.pdf ===
Ошибка чтения PDF: module 'fitz' has no attribute 'open'

=== Обработка файла: 1990-01-11_Правда коммунизма 1990 005.pdf ===
Ошибка чтения PDF: module 'fitz' has no attribute 'open'

=== Обработка фа

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import pymupdf
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПАТТЕРНЫ ДЛЯ ГАЗЕТ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
    "ГАЗЕТА", "Москва", "Москбьі", "Виновников"
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
    "Своем Министерстве", "Министерстве"
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})

# ============================================
# OCR НОРМАЛИЗАЦИЯ
# ============================================

def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))

# ============================================
# ПРЕПРОЦЕССИНГ ГАЗЕТ
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        # >>> CHANGED: нормализуем OCR по строкам до остальной логики
        raw = normalize_ocr_noise(raw)  # построчная нормализация
        line = raw.rstrip()

        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    # склейка переносов слов
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)

    # >>> CHANGED: менее агрессивная нормализация пробелов
    text = re.sub(r"[ \t]+", " ", text)                 # только пробелы/табы
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)              # не схлопывать все абзацы
    text = re.sub(r"[ \t]+\n", "\n", text)

    # >>> CHANGED: глобальную normalize_ocr_noise уже сделали по строкам в merge_lines_gazette_style
    # можно оставить ещё раз, если OCR совсем плохой, но это медленнее
    # text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = []

    for page in doc:
        # вариант: "text" даёт одну строку, близкую к "plain"
        page_text = page.get_text("text")
        raw_pages.append(page_text)

    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)
# ============================================
# ЭВРИСТИКИ ДЛЯ ИМЁН
# ============================================

def looks_like_glued_list(name: str) -> bool:
    """
    Сущность похожа на склеенный список имён:
    слишком много отдельных "фрагментов" с заглавной буквы.
    """
    parts = name.split()
    if len(parts) <= 3:
        return False
    cap_words = sum(1 for p in parts if p and p[0].isupper())
    return cap_words >= 3


def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()
    parts = text.split()

    if looks_like_glued_list(text):
        return True

    # точные плохие одиночные токены
    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    # >>> CHANGED: не вырезаем все однословные имена до 6 букв
    # оставляем только очень короткие (<=3) общие слова
    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 3 and token[0].isupper() and token[1:].islower():
            return True

    # одно длинное слово без суффикса фамилии — похожее на мусор OCR
    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        # >>> CHANGED: подняли порог длины
        if len(only_cyr) >= 10:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                # НЕ режем сразу, это будет учитываться как признак в ML
                # но чтобы не ломать текущую логику — оставим мягкую эвристику:
                # подозрительно длинное слово без фамильного суффикса
                pass

    # подозрительные сочетания букв
    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    # >>> CHANGED: убрали жёсткий запрет на имена со строчной буквы в начале
    # if re.match(r"^[а-яё]", text):
    #     return True

    if " " not in text and len(text) > 24:  # чуть подняли порог
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True


def context_looks_like_index_list(ctx: str) -> bool:
    """
    Отфильтровывает куски, похожие на списки номеров/инициалов
    (много цифр/№/тире, мало букв).
    """
    letters = sum(ch.isalpha() for ch in ctx)
    digits = sum(ch.isdigit() for ch in ctx)
    specials = ctx.count("№") + ctx.count("N") + ctx.count("—") + ctx.count("-")
    length = len(ctx)

    if length == 0:
        return True

    # >>> CHANGED: более мягкое правило, учитывающее доли символов
    letter_ratio = letters / length
    digit_ratio = digits / length if length else 0.0

    # очень мало букв и много цифр/спецсимволов
    if letter_ratio < 0.3 and (digit_ratio > 0.2 or specials > 5):
        return True

    return False


def normalize_name_for_key(name: str) -> str:
    """
    Нормализация имени для дедупликации (грубо: убираем лишние пробелы).
    """
    n = re.sub(r"\.\s+", ". ", name)
    n = re.sub(r"\s{2,}", " ", n)
    return n.strip()

# ============================================
# ML-ФИЛЬТР ИМЁН
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        # >>> NEW: POS‑признаки по самому имени
        doc_name = nlp(name)
        first_pos = doc_name[0].pos_ if len(doc_name) > 0 else "NONE"
        last_pos = doc_name[-1].pos_ if len(doc_name) > 0 else "NONE"
        has_propn = int(any(tok.pos_ == "PROPN" for tok in doc_name))
        has_noun = int(any(tok.pos_ == "NOUN" for tok in doc_name))

        # >>> NEW: признак строчной первой буквы
        starts_with_lower = int(bool(re.match(r"^[а-яё]", name)))

        # >>> NEW: признаки по контексту
        letters_ctx = sum(ch.isalpha() for ch in ctx)
        upp_ctx = sum(ch.isupper() for ch in ctx)
        ctx_len = len(ctx)
        ctx_upper_ratio = upp_ctx / letters_ctx if letters_ctx else 0.0

        feat: Dict[str, Any] = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),

            # >>> NEW:
            "first_pos": first_pos,
            "last_pos": last_pos,
            "has_propn": has_propn,
            "has_noun": has_noun,
            "starts_with_lower": starts_with_lower,

            "ctx_len": ctx_len,
            "ctx_upper_ratio": ctx_upper_ratio,
            "method": cand.method,
        }

        # >>> NEW: контекстные ключевые слова (старые признаки оставляем)
        feat["ctx_has_pozicion"] = int(bool(re.search(
            r"директор|секретарь|председатель|первый секретарь|депутат",
            ctx, re.I
        )))
        feat["ctx_has_prof"] = int(bool(re.search(
            r"доярк|плавильщик|слесарь|редактор|журналист|учитель|врач|агроном",
            ctx, re.I
        )))

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=300, class_weight="balanced")  # чуть больше итераций

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()

# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(
    text: str,
    use_trained_filter: bool = True,
    proba_threshold: float = 0.5  # базовый порог
) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            # "А.", "А. Б."
            if re.fullmatch(r"[А-ЯЁ]\.(\s[А-ЯЁ]\.)?", name):
                continue
            # обрубки типа "А. Ш"
            if re.fullmatch(r"[А-ЯЁ]\.\s*[А-ЯЁ]$", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            if bad_ctx_re.search(sent.text):
                continue

            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)

            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            if bad_ctx_re.search(ctx_text):
                continue
            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            if context_looks_like_index_list(ctx_text):
                continue
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        clean_name = normalize_name_for_key(clean_name)
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered: List[Tuple[str, str, str]] = []
        for cand, p in zip(final_candidates, proba):
            # >>> NEW: разные пороги для разных методов
            thr = proba_threshold
            if cand.method.startswith("regex"):
                thr = max(thr, 0.6)
            if cand.method == "signature":
                thr = min(thr, 0.4)
            if p >= thr:
                filtered.append((cand.name, cand.ctx, cand.method))
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered

# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ JSON
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)

# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 1. Путь к размеченному JSON
    labeled_json = r"C:\Users\nikol\Desktop\disser\labeled_names.json"  # поправь при необходимости

    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)
    else:
        print(f"Внимание: файл разметки не найден: {labeled_json}")
        print("Фильтр имён будет работать только на эвристиках (без ML).")

    # 2. Обработка папки с pdf-газетами
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)

=== NameFilterModel report ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         5
           1      1.000     1.000     1.000         8

    accuracy                          1.000        13
   macro avg      1.000     1.000     1.000        13
weighted avg      1.000     1.000     1.000        13


=== Обработка файла: 1990-01-01_Правда коммунизма 1990 001.pdf ===
Найдено уникальных имён: 82
[spacy] Михаил Вандышев :: Среди женщмн-олераторов машинного доения на районных соревнованиях он, дояр из Клевакино, Михаил Вандышев замечательно смотрится. М ежду прочим.
------------------------------------------------------------
[spacy] реж евлян :: И все же больше их вклад в городскую копилку, уровень жизни реж евлян зависит от этих коллективов в первую очередь. Итак, два вопроса, новогодней анкеты трем директорам.
------------------------------------------------------------
[spacy] дем неремен :: Сейчас вступаем в предвыборную кампани

KeyboardInterrupt: 

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import pymupdf  
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПРЕПРОЦЕССИНГ ТЕКСТА ПОД ГАЗЕТЫ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})


def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))


def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    # OCR нормализация уже над чистым текстом
    text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)


# ============================================
# ФИЛЬТРАЦИЯ ИМЁН
# ============================================

def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()

    # Точно плохие одиночные токены
    parts = text.split()
    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    # Одно короткое слово: "Мой", "Ваши" и т.п.
    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 6 and token[0].isupper() and token[1:].islower():
            return True

    # Одно длинное слово без суффикса фамилии — похожее на мусор OCR
    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        if len(only_cyr) >= 7:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                return True

    # Подозрительные сочетания букв
    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    # Старая твоя логика:
    if re.match(r"^[а-яё]", text):
        return True

    if " " not in text and len(text) > 18:
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True


# ============================================
# ДОOБУЧАЕМЫЙ ФИЛЬТР ИМЁН
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    Используем sklearn Pipeline: DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        # простые символьные признаки
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        feat = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),
            "ctx_has_pozicion": int(bool(re.search(r"директор|секретарь|председатель|первый секретарь|депутат", ctx, re.I))),
            "ctx_has_prof": int(bool(re.search(r"доярк|плавильщик|слесарь|редактор|журналист|учитель", ctx, re.I))),
        }

        # POS внутри сущности (если есть span)
        if cand.sent is not None:
            # найдём span по тексту (грубо)
            try:
                doc = cand.sent.doc
                # первый токен совпавший по тексту
                for tok in cand.sent:
                    if tok.text in tokens[0]:
                        # посчитаем POS вокруг
                        break
                # чтобы не усложнять: не используем точный span; просто нет POS‑фич
            except Exception:
                pass

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        """
        candidates: список CandidateName
        labels: 1 — настоящее имя, 0 — шум
        """
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=200, class_weight="balanced")

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            # если модель не обучена — пускаем всё
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        # вероятность класса 1 (real person)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()


# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(text: str,
                    use_trained_filter: bool = True,
                    proba_threshold: float = 0.5) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            # контекстные стоп‑фразы
            if bad_ctx_re.search(ent.sent.text):
                continue

            sent = ent.sent
            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)
            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            # контекстные стоп‑фразы
            if bad_ctx_re.search(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» в конце абзаца ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered = [
            (cand.name, cand.ctx, cand.method)
            for cand, p in zip(final_candidates, proba)
            if p >= proba_threshold
        ]
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered


# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ КОРПУСУ
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    """
    Ожидаемый формат JSON:
    [
      {
        "name": "А. Шангина",
        "ctx": "Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ? ...",
        "method": "spacy",
        "label": 1
      },
      {
        "name": "Мой",
        "ctx": "На Мой взгляд долго мы будем чувствовать...",
        "method": "spacy",
        "label": 0
      },
      ...
    ]
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)


# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 1. Дообучаем фильтр на разметке (один раз)
    labeled_json = r"C:\path\to\labeled_names.json"
    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)

    # 2. Обрабатываем газеты
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)


=== Обработка файла: 1990-01-01_Правда коммунизма 1990 001.pdf ===
Найдено уникальных имён: 51
[spacy] Михаил Вандышев :: Среди женщмн-олераторов машинного доения на районных соревнованиях он, дояр из Клевакино, Михаил Вандышев замечательно смотрится. М ежду прочим.
------------------------------------------------------------
[spacy] А. Шангина :: Ф ото А. Шангина. КАК СТАТЬ ХОЗЯИНОМ?
------------------------------------------------------------
[spacy] Любови Михайловны Тоіпоркозой :: Неужели в веірху сидящ ие не понимают, что и сами могут оказаться в глазном отделении, оказаться в таких умелых и добрых руках, как у Любови Михайловны Тоіпоркозой. Скрро 'начнутся...
------------------------------------------------------------
[spacy] Кузьма Тимофеевич Швецов :: Мой собеседник Кузьма Тимофеевич Швецов родился в Ощепково, когда взошла заря Красного Октября. Именно так и был впоследствии назван колхоз в этой деревне.
------------------------------------------------------------
[spacy] Куз

In [21]:
import os
import re
import json
from pathlib import Path
from typing import List, Dict, Any
from collections import Counter
from datetime import datetime

import pandas as pd
from tqdm import tqdm
import nltk
nltk.download('punkt', quiet=True)

# Лемматизация русскоязычных текстов
try:
    import pymorphy2
    morph = pymorphy2.MorphAnalyzer()
except Exception:
    morph = None

def normalize_text(text: str) -> str:
    """
    Нормализация текста: приведение к нижнему регистру,
    удаление лишних пробелов, замена дефисов, удаление управляющих символов.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)  # схлопывание пробелов
    text = text.strip()
    return text

def tokenize_text(text: str) -> List[str]:
    """
    Токенизация текста на слова с использованием nltk.
    Убираются пункты и числа.
    """
    if not text:
        return []
    # простой разбиение на слова
    tokens = nltk.word_tokenize(text, language="russian")
    # фильтрация: только алфавитные и числовые токены, удаление коротких
    tokens = [t for t in tokens if re.match(r"^\w+$", t)]
    return tokens

def lemmatize_token(token: str) -> str:
    """
    Лемматизация одного токена с использованием pymorphy2, если доступна.
    Возвращает базовую форму слова.
    """
    if morph is None:
        return token
    parsed = morph.parse(token)
    if not parsed:
        return token
    # выбираем наиболее вероятную форму
    return parsed[0].normal_form

def lemmatize_tokens(tokens: List[str]) -> List[str]:
    if morph is None:
        return tokens
    return [lemmatize_token(t) for t in tokens]

def parse_article_entry(entry: Dict[str, Any]) -> Dict[str, Any]:
    """
    Преобразование одной статьи из формата входных данных в унифицированный словарь.
    Ожидаются поля:
      - date: строка с датой или числовой timestamp
      - title: заголовок статьи
      - text: текст статьи
      - other_meta: прочие метаданные
    """
    date_raw = entry.get("date") or entry.get("publish_date") or entry.get("year")
    title = entry.get("title") or ""
    text = entry.get("text") or ""

    # Попытка привести дату к формату YYYY-MM-DD
    date_str = None
    if isinstance(date_raw, str):
        # попытки распознать даты в нескольких форматах
        for fmt in ("%Y-%m-%d", "%d.%m.%Y", "%d/%m/%Y", "%Y/%m/%d"):
            try:
                dt = datetime.strptime(date_raw.strip(), fmt)
                date_str = dt.strftime("%Y-%m-%d")
                break
            except Exception:
                continue
    elif isinstance(date_raw, (int, float)):
        try:
            dt = datetime.fromtimestamp(float(date_raw))
            date_str = dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    if date_str is None:
        # попытка извлечь год
        m = re.search(r"\d{4}", str(date_raw))
        if m:
            date_str = f"{m.group(0)}-01-01"

    return {
        "date": date_str,
        "title": title,
        "text": text,
        "raw": entry
    }

def load_articles_from_folder(folder_path: str) -> List[Dict[str, Any]]:
    """
    Загружает статьи из файлов в папке.
    Ожидается, что каждый файл содержит JSON-объект или массив объектов.
    Альтернативно поддерживается простой .jsonl формат.
    """
    articles = []
    p = Path(folder_path)
    if not p.exists():
        return articles

    # поддерживаем .json, .jsonl, и текстовые файлы с простой структурой
    for fp in tqdm(list(p.rglob("*")):
        pass


def load_articles_from_folder(folder_path: str) -> List[Dict[str, Any]]:
    articles = []
    p = Path(folder_path)
    if not p.exists():
        return articles

    for file in tqdm(list(p.rglob("*"))):
        if file.is_file():
            try:
                if file.suffix.lower() in {".json"}:
                    with file.open("r", encoding="utf-8") as f:
                        data = json.load(f)
                        if isinstance(data, list):
                            for item in data:
                                articles.append(parse_article_entry(item))
                        elif isinstance(data, dict):
                            articles.append(parse_article_entry(data))
                elif file.suffix.lower() in {".jsonl", ".txt"}:
                    with file.open("r", encoding="utf-8") as f:
                        for line in f:
                            line = line.strip()
                            if not line:
                                continue
                            obj = json.loads(line)
                            articles.append(parse_article_entry(obj))
            except Exception as e:
                # пропустим проблемы с конкретным файлом
                continue
    return articles

def clean_texts_and_build_vocab(articles: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Очистка текстов и построение словаря-частотности.
    Возвращает словарь с частотами и упорядоченным списком слов.
    """
    all_tokens: List[str] = []
    vocab_counter = Counter()

    for art in articles:
        text = art.get("text", "")
        text_norm = normalize_text(text)
        tokens = tokenize_text(text_norm)
        tokens = lemmatize_tokens(tokens)
        # фильтрация очень коротких токенов
        tokens = [t for t in tokens if len(t) > 2]
        vocab_counter.update(tokens)
        art["tokens"] = tokens
        all_tokens.extend(tokens)

    vocab = [w for w, _ in vocab_counter.most_common()]
    return {
        "articles": articles,
        "vocab": vocab,
        "counter": vocab_counter,
        "total_tokens": len(all_tokens)
    }

def save_results(results: Dict[str, Any], out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    # сохранить статистику
    stats = {
        "total_articles": len(results["articles"]),
        "total_tokens": results["total_tokens"],
        "vocabulary_size": len(results["vocab"])
    }
    with open(os.path.join(out_dir, "stats.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    # сохранить статьи с добавленными токенами
    with open(os.path.join(out_dir, "articles_with_tokens.json"), "w", encoding="utf-8") as f:
        json.dump([{
            "date": a.get("date"),
            "title": a.get("title"),
            "text": a.get("text"),
            "tokens": a.get("tokens", [])
        } for a in results["articles"]], f, ensure_ascii=False, indent=2)

    # сохранение словаря в формате txt
    with open(os.path.join(out_dir, "vocabulary.txt"), "w", encoding="utf-8") as f:
        for w in results["vocab"]:
            f.write(w + "\n")

def main():
    # Настройки
    input_folder = "data/gazety"  # заменить на путь к коллекции
    out_folder = "output/gazety_processing"

    # Загрузка
    articles_raw = load_articles_from_folder(input_folder)

    # Преобразование и чистка
    results = clean_texts_and_build_vocab(articles_raw)

    # Сохранение
    save_results(results, out_folder)
    print("Обработка завершена. Результаты сохранены в:", out_folder)

if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (2299892456.py, line 120)

In [ ]:
import os
import re
import json
import logging
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Any, Optional
from collections import defaultdict

import pymupdf  # fitz
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ====================================================
# ЛОГИРОВАНИЕ
# ====================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


# ====================================================
# КОНФИГ
# ====================================================

class GazetteConfig:
    HEADER_PATTERNS = [
        r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ",
        r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС",
        r"Газета основана",
        r"ПОДПИСНОЙ ИНДЕКС",
        r"ЦЕНА \d+ КОП",
    ]

    FOOTER_PATTERNS = [
        r"АДРЕС РЕДАКЦИИ",
        r"ПИШИТЕ,\s*ЗАХОДИТЕ",
        r"ТИРАЖ",
        r"ЗВОНИТЕ",
        r"РЕДАКЦИОННЫЙ СОВЕТ",
        r"РЕДАКТОР",
        r"Индекс\s+\d+",
        r"ул\.\s*Красноармейская",
        r"Редактор",
        r"тираж",
    ]

    TV_SECTION_PATTERNS = [
        r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ",
        r"ТЕЛЕПРОГРАММА",
        r"ПОНЕДЕЛЬНИК",
        r"ВТОРНИК",
        r"СРЕДА",
        r"ЧЕТВЕРГ",
        r"ПЯТНИЦА",
        r"СУББОТА",
        r"ВОСКРЕСЕНЬЕ",
    ]

    NON_PERSON_TOKENS = {
        "Правда", "Коммунизма", "ПРАВДА", "КОММУНИЗМА",
        "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
        "Совет", "Совета", "СССР", "РСФСР", "РС", "Ф", "С", "Р",
        "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
        "Свердловского", "облисполкома",
        "Реж", "Режевского", "Режевский", "Режевская", "Режский",
        "Глинский", "Глинская", "Глинское",
        "Совхоз", "СОВХОЗ", "Райком", "Горком",
        "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
        "СТРАНИЦА", "РЕПОРТЕРА", "Газета", "ГАЗЕТА",
        "Советов", "Время", "Боевой", "Комм", "Брак",
        "ПАРТИИ", "ПАРТИЯ", "КОМСОМОЛ", "ПИОНЕР",
        "КОЛХОЗ", "СОВХОЗ", "ЗАВОД", "ФАБРИКА",
    }

    RUSSIAN_SURNAME_SUFFIXES = (
        "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
        "ский", "ская", "ской", "ское", "ских",
        "овский", "евский", "инский", "ынский",
        "ой", "ая", "ий", "ое",
    )

    MIN_NAME_LENGTH = 2
    MAX_NAME_LENGTH = 50

    BAD_TOKENS_IN_NAMES = {"Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо"}


# ====================================================
# НОРМАЛИЗАЦИЯ OCR
# ====================================================

class TextNormalizer:
    LAT_TO_CYR_TABLE = str.maketrans({
        "A": "А", "a": "а",
        "B": "В", "b": "ь",  # b часто даёт мягкий знак / b
        "C": "С", "c": "с",
        "E": "Е", "e": "е",
        "H": "Н", "h": "н",
        "K": "К", "k": "к",
        "M": "М", "m": "м",
        "O": "О", "o": "о",
        "P": "Р", "p": "р",
        "T": "Т", "t": "т",
        "X": "Х", "x": "х",
        "Y": "У", "y": "у",
    })

    @staticmethod
    def normalize_ocr_text(text: str) -> str:
        if not text:
            return ""

        # Склейка переносов через дефис
        text = re.sub(r"(\b\w+)-\s*\n\s*(\w+\b)", r"\1\2", text)

        # Склейка очевидных разрывов внутри слова
        text = re.sub(r"(\b[А-Яа-яЁё])\s+([А-Яа-яЁё]+\b)", r"\1\2", text)

        # Латиницу → кириллицу
        text = text.translate(TextNormalizer.LAT_TO_CYR_TABLE)

        # Частые OCR-ошибки (примерные)
        corrections = {
            r"\b([Пп])[рp](\w+)": r"\1р\2",     # пpавда → правда
            r"\b([Вв])o(\w+)": r"\1о\2",       # вoда → вода
            r"\b([Нн])a(\w+)": r"\1а\2",       # наc → нас
        }
        for pat, repl in corrections.items():
            text = re.sub(pat, repl, text)

        return text

    @staticmethod
    def fix_spaced_caps(line: str) -> str:
        def repl(m):
            return m.group(0).replace(" ", "")
        return re.sub(r"((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])", repl, line)

    @staticmethod
    def merge_hyphenated_words(line: str) -> str:
        return re.sub(r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)", r"\1\2", line)

    @staticmethod
    def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
        if not line:
            return True
        cleaned = re.sub(r"\s+", "", line)
        if not cleaned:
            return True
        noise_chars = sum(
            1 for ch in cleaned
            if not ch.isalnum() and ch not in ".,-—:;!?\"()"
        )
        return noise_chars / len(cleaned) > threshold


# ====================================================
# PDF → ЧИСТЫЙ ТЕКСТ
# ====================================================

class PDFProcessor:
    def __init__(self, config: GazetteConfig | None = None):
        self.config = config or GazetteConfig()
        self.header_re = re.compile("|".join(self.config.HEADER_PATTERNS), re.IGNORECASE)
        self.footer_re = re.compile("|".join(self.config.FOOTER_PATTERNS), re.IGNORECASE)
        self.tv_re = re.compile("|".join(self.config.TV_SECTION_PATTERNS), re.IGNORECASE)

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(pdf_path)

        doc = pymupdf.open(pdf_path)
        pages = []
        for page in doc:
            t = page.get_text()
            # режем шапки/подвалы/телепрограмму по страницам
            t = self.header_re.sub("", t)
            t = self.footer_re.sub("", t)
            t = self.tv_re.sub("", t)
            pages.append(t)
        doc.close()

        raw = "\n".join(pages)
        raw = TextNormalizer.normalize_ocr_text(raw)
        return self._master_clean_text(raw)

    def _master_clean_text(self, raw_text: str) -> str:
        # Склейка переносов
        text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
        text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
        text = re.sub(r"[«»“”]", '"', text)

        lines = text.split("\n")
        cleaned_lines: List[str] = []
        for line in lines:
            l = line.rstrip()

            if self.header_re.search(l) or self.footer_re.search(l) or self.tv_re.search(l):
                continue
            if TextNormalizer.is_mostly_noise(l):
                continue
            cleaned_lines.append(l)

        merged = self._merge_lines_gazette_style(cleaned_lines)
        text = "\n\n".join(merged)

        text = re.sub(r"\s+([.,!?;:])", r"\1", text)
        text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)

        return text.strip()

    def _merge_lines_gazette_style(self, lines: List[str]) -> List[str]:
        merged: List[str] = []
        prev = ""

        def is_probable_heading(line: str) -> bool:
            txt = line.strip()
            if len(txt) < 4 or len(txt) > 70:
                return False
            upp = sum(1 for ch in txt if ch.isupper())
            letters = sum(1 for ch in txt if ch.isalpha())
            if letters == 0:
                return False
            return upp / letters > 0.8

        for raw in lines:
            line = raw.rstrip()
            if not line.strip():
                if prev:
                    merged.append(prev.strip())
                    prev = ""
                continue

            if TextNormalizer.is_mostly_noise(line):
                continue

            line = TextNormalizer.fix_spaced_caps(line)
            line = TextNormalizer.merge_hyphenated_words(line)

            if not prev:
                prev = line.strip()
                continue

            if is_probable_heading(line):
                merged.append(prev.strip())
                prev = line.strip()
                continue

            # если пред строка не заканчивается знаком конца предложения — сливаем
            if not re.search(r"[.!?…»)]\s*$", prev):
                prev = prev + " " + line.strip()
            else:
                merged.append(prev.strip())
                prev = line.strip()

        if prev:
            merged.append(prev.strip())
        return merged


# ====================================================
# ВАЛИДАЦИЯ ИМЁН
# ====================================================

class NameValidator:
    @staticmethod
    def looks_like_russian_name(name: str) -> bool:
        if not name:
            return False
        name = name.strip()
        if len(name) < GazetteConfig.MIN_NAME_LENGTH or len(name) > GazetteConfig.MAX_NAME_LENGTH:
            return False
        if not name[0].isupper():
            return False

        parts = name.split()
        for part in parts:
            if re.search(r"[#@$%^&*|/\\\[\]{}<>]", part):
                return False
            if part in GazetteConfig.BAD_TOKENS_IN_NAMES:
                return False
            if part in GazetteConfig.NON_PERSON_TOKENS:
                return False

        # 3 слова: Фам Имя Отч
        if len(parts) == 3:
            surname = parts[0]
            if any(surname.lower().endswith(s) for s in GazetteConfig.RUSSIAN_SURNAME_SUFFIXES):
                return True
            return False

        # Фамилия И.О.
        if len(parts) == 2:
            a, b = parts
            if any(a.lower().endswith(s) for s in GazetteConfig.RUSSIAN_SURNAME_SUFFIXES) and \
                    re.fullmatch(r"[А-ЯЁ]\.\s*[А-ЯЁ]\.", b):
                return True

        # И.О. Фамилия
        if len(parts) == 2:
            a, b = parts
            if re.fullmatch(r"[А-ЯЁ]\.\s*[А-ЯЁ]\.", a) and \
                    any(b.lower().endswith(s) for s in GazetteConfig.RUSSIAN_SURNAME_SUFFIXES):
                return True

        # Одна фамилия
        if len(parts) == 1:
            s = parts[0]
            if len(s) >= 3 and any(s.lower().endswith(x) for x in GazetteConfig.RUSSIAN_SURNAME_SUFFIXES):
                return True

        return False

    @staticmethod
    def is_garbage_name(name: str) -> bool:
        low = name.lower()

        # много согласных подряд
        if re.search(r"[бвгджзклмнпрстфхцчшщ]{5,}", low):
            return True
        # много гласных подряд
        if re.search(r"[аеёиоуыэюя]{4,}", low):
            return True

        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in name)
        lat = sum("A" <= ch <= "Z" or "a" <= ch <= "z" for ch in name)
        if cyr and lat:
            return True

        if len(name.replace(".", "")) > 25 and " " not in name:
            return True

        return False


# ====================================================
# ДАТАКЛАСС ДЛЯ УПОМИНАНИЙ
# ====================================================

@dataclass
class PersonMention:
    name: str
    context: str
    extraction_method: str
    confidence: float = 1.0
    position: Optional[Tuple[int, int]] = None
    metadata: Optional[Dict[str, Any]] = None

    def to_dict(self) -> Dict[str, Any]:
        d = asdict(self)
        if self.position is not None:
            d["position"] = [self.position[0], self.position[1]]
        return d


# ====================================================
# ML ФИЛЬТР (ОПЦИОНАЛЬНО)
# ====================================================

class NameMLFilter:
    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def train(self, labeled_items: List[Dict[str, Any]]) -> None:
        X = [self._extract_features(item["name"], item["ctx"]) for item in labeled_items]
        y = [int(item["label"]) for item in labeled_items]

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=500, class_weight="balanced")

        self.pipeline = Pipeline([("vec", vec), ("clf", clf)])

        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        self.pipeline.fit(Xtr, ytr)
        ypred = self.pipeline.predict(Xte)
        logger.info("=== NameMLFilter report ===\n" + classification_report(yte, ypred, digits=3))

    def predict_proba(self, mention: PersonMention) -> float:
        if not self.pipeline:
            return 1.0
        feat = self._extract_features(mention.name, mention.context)
        proba = self.pipeline.predict_proba([feat])[0]
        return float(proba[1])

    def _extract_features(self, name: str, ctx: str) -> Dict[str, Any]:
        parts = name.split()
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        f = {
            "n_tokens": len(parts),
            "name_len": len(name),
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "has_dot": int("." in name),
            "has_hyphen": int("-" in name),
            "garbage": int(NameValidator.is_garbage_name(name)),
            "has_surname_suffix": int(
                any(only_cyr.lower().endswith(s) for s in GazetteConfig.RUSSIAN_SURNAME_SUFFIXES)
            ),
        }

        low_ctx = ctx.lower()
        f["ctx_has_title"] = int(bool(re.search(
            r"директор|секретарь|председатель|депутат|начальник|заведующ", low_ctx
        )))
        f["ctx_has_prof"] = int(bool(re.search(
            r"доярк|плавильщик|слесар|редактор|журналист|учител|врач|инженер|рабоч", low_ctx
        )))
        return f


# ====================================================
# ОСНОВНОЙ ЭКСТРАКТОР ИМЁН
# ====================================================

class GazettePersonExtractor:
    def __init__(self, spacy_model: str = "ru_core_news_lg", ml_filter: NameMLFilter | None = None):
        try:
            self.nlp = spacy.load(spacy_model)
            logger.info(f"spaCy модель загружена: {spacy_model}")
        except OSError:
            logger.warning(f"Модель {spacy_model} не найдена. Загружаю blank('ru')")
            self.nlp = spacy.blank("ru")

        self.validator = NameValidator()
        self.ml_filter = ml_filter

        self.patterns = {
            "fio": re.compile(r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b"),
            "f_iio": re.compile(r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b"),
            "iio_f": re.compile(r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b"),
            "i_f": re.compile(r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b"),
            "signature": re.compile(r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$", re.MULTILINE),
        }

    def extract_persons(self, text: str, use_ml: bool = True, proba_threshold: float = 0.5) -> List[PersonMention]:
        if not text:
            return []

        cands: List[PersonMention] = []
        cands.extend(self._extract_with_spacy(text))
        cands.extend(self._extract_with_regex(text))
        cands.extend(self._extract_signatures(text))

        # дедупликация + объединение методов
        final = self._filter_and_deduplicate(cands)

        # ML фильтр
        if use_ml and self.ml_filter is not None:
            filtered: List[PersonMention] = []
            for m in final:
                p = self.ml_filter.predict_proba(m)
                if p >= proba_threshold:
                    m.confidence = p
                    filtered.append(m)
            return filtered
        else:
            return final

    def _extract_with_spacy(self, text: str) -> List[PersonMention]:
        mentions: List[PersonMention] = []
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if not self.validator.looks_like_russian_name(name):
                continue
            if self.validator.is_garbage_name(name):
                continue
            ctx = self._sent_plus_next(doc, ent.sent)
            mentions.append(PersonMention(
                name=name,
                context=ctx,
                extraction_method="spacy",
                confidence=0.9,
                position=(ent.start_char, ent.end_char),
            ))
        return mentions

    def _sent_plus_next(self, doc, sent_span) -> str:
        sents = list(doc.sents)
        try:
            idx = sents.index(sent_span)
        except ValueError:
            return sent_span.text.strip()
        ctx_sents = [sents[idx].text.strip()]
        if idx + 1 < len(sents):
            ctx_sents.append(sents[idx + 1].text.strip())
        ctx = " ".join(ctx_sents)
        ctx = re.sub(r"\s+", " ", ctx).strip()
        return ctx

    def _extract_with_regex(self, text: str) -> List[PersonMention]:
        mentions: List[PersonMention] = []
        for name_pat, pat in self.patterns.items():
            if name_pat == "signature":
                continue
            for m in pat.finditer(text):
                name = m.group(0).strip()
                if not self.validator.looks_like_russian_name(name):
                    continue
                if self.validator.is_garbage_name(name):
                    continue
                start, end = m.span()
                ctx_start = max(0, start - 100)
                ctx_end = min(len(text), end + 200)
                ctx = text[ctx_start:ctx_end]
                ctx = re.sub(r"\s+", " ", ctx).strip()
                mentions.append(PersonMention(
                    name=name,
                    context=ctx,
                    extraction_method=f"regex_{name_pat}",
                    confidence=0.8,
                    position=(start, end),
                ))
        return mentions

    def _extract_signatures(self, text: str) -> List[PersonMention]:
        mentions: List[PersonMention] = []
        paragraphs = [p for p in re.split(r"\n{2,}", text) if p.strip()]
        for i, para in enumerate(paragraphs):
            for m in self.patterns["signature"].finditer(para):
                # подпись обычно вида "А. Фамилия"
                name = m.group(0).strip()
                # но мы хотим именно "А. Фамилия"
                # можно взять только первые два токена
                tokens = name.split()
                short = " ".join(tokens[:2])
                if not self.validator.looks_like_russian_name(short):
                    continue
                ctx_parts = []
                if i > 0:
                    ctx_parts.append(paragraphs[i - 1])
                ctx_parts.append(para)
                if i + 1 < len(paragraphs):
                    ctx_parts.append(paragraphs[i + 1])
                ctx = "\n\n".join(ctx_parts)
                ctx = re.sub(r"\s+", " ", ctx).strip()
                mentions.append(PersonMention(
                    name=short,
                    context=ctx,
                    extraction_method="signature",
                    confidence=0.7,
                ))
        return mentions

    def _filter_and_deduplicate(self, mentions: List[PersonMention]) -> List[PersonMention]:
        groups: Dict[str, List[PersonMention]] = defaultdict(list)
        for m in mentions:
            norm_name = re.sub(r"\s+", " ", m.name).strip().rstrip(".")
            groups[norm_name].append(m)

        final: List[PersonMention] = []
        for name, group in groups.items():
            group.sort(key=lambda x: x.confidence, reverse=True)
            best = group[0]
            if len(group) > 1:
                all_methods = sorted({g.extraction_method for g in group})
                best.metadata = {
                    "all_extraction_methods": all_methods,
                    "total_mentions": len(group),
                }
            best.name = name
            final.append(best)
        return final


# ====================================================
# ОБРАБОТКА ПАПКИ
# ====================================================

def process_gazette_folder(
    input_folder: str,
    output_folder: Optional[str] = None,
    ml_filter: Optional[NameMLFilter] = None,
) -> Dict[str, Any]:
    if not os.path.isdir(input_folder):
        raise FileNotFoundError(input_folder)

    if output_folder is None:
        output_folder = os.path.join(input_folder, "processed")
    os.makedirs(output_folder, exist_ok=True)

    pdf_proc = PDFProcessor()
    extractor = GazettePersonExtractor(ml_filter=ml_filter)

    files = [f for f in os.listdir(input_folder) if f.lower().endswith(".pdf")]
    logger.info(f"Найдено PDF: {len(files)}")

    results: Dict[str, Any] = {}

    for fname in sorted(files):
        path = os.path.join(input_folder, fname)
        logger.info(f"Обработка: {fname}")
        try:
            text = pdf_proc.extract_text_from_pdf(path)
            persons = extractor.extract_persons(text, use_ml=(ml_filter is not None))

            out = {
                "filename": fname,
                "text_length": len(text),
                "persons_found": len(persons),
                "persons": [p.to_dict() for p in persons],
            }

            base = os.path.splitext(fname)[0]
            out_path = os.path.join(output_folder, f"{base}_persons.json")
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(out, f, ensure_ascii=False, indent=2)

            results[fname] = {
                "persons_count": len(persons),
                "output_file": out_path,
            }
            logger.info(f"  имён: {len(persons)}, файл: {out_path}")
        except Exception as e:
            logger.exception(f"Ошибка для {fname}: {e}")
            results[fname] = {"error": str(e)}

    summary = {
        "total_files": len(files),
        "successfully_processed": sum(1 for r in results.values() if "persons_count" in r),
        "total_persons": sum(r.get("persons_count", 0) for r in results.values()),
        "details": results,
    }
    summary_path = os.path.join(output_folder, "processing_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    logger.info(f"Сводка: {summary_path}")
    return summary


# ====================================================
# CLI
# ====================================================

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Извлечение имён из советских газет (PDF)")
    parser.add_argument("input_folder", help="Папка с PDF")
    parser.add_argument("--output", help="Папка для результатов (по умолчанию input_folder/processed)")
    parser.add_argument("--train", help="JSON с размеченными именами для ML-фильтра")
    args = parser.parse_args()

    ml_filter = None
    if args.train and os.path.exists(args.train):
        logger.info(f"Обучение ML-фильтра по {args.train}")
        with open(args.train, "r", encoding="utf-8") as f:
            labeled = json.load(f)
        ml_filter = NameMLFilter()
        ml_filter.train(labeled)

    summary = process_gazette_folder(args.input_folder, args.output, ml_filter=ml_filter)

    print()
    print("=== Готово ===")
    print(f"Файлов: {summary['total_files']}")
    print(f"Успешно: {summary['successfully_processed']}")
    print(f"Всего упоминаний людей: {summary['total_persons']}")

usage: ipykernel_launcher.py [-h] [--output OUTPUT] [--train TRAIN]
                             input_folder
ipykernel_launcher.py: error: the following arguments are required: input_folder


SystemExit: 2

c:\Users\nikol\Desktop\disser\.venv-nlp\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import pymupdf  # from PyMuPDF
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПАТТЕРНЫ ДЛЯ ГАЗЕТ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
    "ГАЗЕТА", "Москва", "Москбьі", "Виновников"
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
    "Своем Министерстве", "Министерстве"
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})

# ============================================
# OCR НОРМАЛИЗАЦИЯ
# ============================================

def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))

# ============================================
# ПРЕПРОЦЕССИНГ ГАЗЕТ
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)

# ============================================
# ЭВРИСТИКИ ДЛЯ ИМЁН
# ============================================

def looks_like_glued_list(name: str) -> bool:
    """
    Сущность похожа на склеенный список имён:
    слишком много отдельных "фрагментов" с заглавной буквы.
    """
    parts = name.split()
    if len(parts) <= 4:
        return False
    cap_words = sum(1 for p in parts if p and p[0].isupper())
    return cap_words >= 3


def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()
    parts = text.split()

    if looks_like_glued_list(text):
        return True

    # точные плохие одиночные токены
    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    # одно короткое слово: "Мой", "Ваши" и т.п.
    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 6 and token[0].isupper() and token[1:].islower():
            return True

    # одно длинное слово без суффикса фамилии — похожее на мусор OCR
    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        if len(only_cyr) >= 7:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                return True

    # подозрительные сочетания букв
    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    # старая логика:
    if re.match(r"^[а-яё]", text):
        return True

    if " " not in text and len(text) > 18:
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True


def context_looks_like_index_list(ctx: str) -> bool:
    """
    Отфильтровывает куски, похожие на списки номеров/инициалов
    (много цифр/№/тире, мало букв).
    """
    letters = sum(ch.isalpha() for ch in ctx)
    digits = sum(ch.isdigit() for ch in ctx)
    specials = ctx.count("№") + ctx.count("N") + ctx.count("—") + ctx.count("-")
    length = len(ctx)

    if length == 0:
        return True

    if letters < 10 and (digits + specials) > 10:
        return True

    return False


def normalize_name_for_key(name: str) -> str:
    """
    Нормализация имени для дедупликации (грубо: убираем лишние пробелы).
    """
    n = re.sub(r"\.\s+", ". ", name)
    n = re.sub(r"\s{2,}", " ", n)
    return n.strip()

# ============================================
# ML-ФИЛЬТР ИМЁН
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        feat = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),
            "ctx_has_pozicion": int(bool(re.search(
                r"директор|секретарь|председатель|первый секретарь|депутат",
                ctx, re.I
            ))),
            "ctx_has_prof": int(bool(re.search(
                r"доярк|плавильщик|слесарь|редактор|журналист|учитель|врач|агроном",
                ctx, re.I
            ))),
        }

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=200, class_weight="balanced")

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()

# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(
    text: str,
    use_trained_filter: bool = True,
    proba_threshold: float = 0.5
) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            if bad_ctx_re.search(sent.text):
                continue

            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)

            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            if bad_ctx_re.search(ctx_text):
                continue
            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            if context_looks_like_index_list(ctx_text):
                continue
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        clean_name = normalize_name_for_key(clean_name)
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered = [
            (cand.name, cand.ctx, cand.method)
            for cand, p in zip(final_candidates, proba)
            if p >= proba_threshold
        ]
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered

# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ JSON
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)

# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 1. Путь к размеченному JSON
    labeled_json = r"C:\Users\nikol\Desktop\disser\labeled_names.json"  # поправь при необходимости

    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)
    else:
        print(f"Внимание: файл разметки не найден: {labeled_json}")
        print("Фильтр имён будет работать только на эвристиках (без ML).")

    # 2. Обработка папки с pdf-газетами
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Dict, Any, Optional

import sqlite3  # <-- добавили

import pymupdf  # from PyMuPDF
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ============================================
# ПУТЬ К БАЗЕ ДАННЫХ
# ============================================

DB_PATH = "newspapers.db"


def create_database():
    """
    Создаёт базу данных и таблицу newspaper_mentions.
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)

    conn.commit()
    conn.close()
    print(f"База данных '{DB_PATH}' и таблица 'newspaper_mentions' созданы.")


def insert_mentions(
    newspaper_name: str,
    publication_date: str,
    pdf_filename: str,
    persons: List[Tuple[str, str, str]]
):
    """
    Сохраняет найденные упоминания персон в таблицу newspaper_mentions.
    persons: список кортежей (name, ctx, method), как возвращает extract_persons.
    """
    if not persons:
        return

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    rows = [
        (newspaper_name, publication_date, name, ctx, pdf_filename)
        for (name, ctx, _method) in persons
    ]

    cursor.executemany("""
        INSERT INTO newspaper_mentions (
            newspaper_name,
            publication_date,
            person_name,
            context,
            pdf_filename
        ) VALUES (?, ?, ?, ?, ?)
    """, rows)

    conn.commit()
    conn.close()
    print(f"В БД добавлено {len(rows)} записей для файла {pdf_filename}.")


# ============================================
# ЗАГРУЗКА МОДЕЛИ SPACY
# ============================================

# Требуется установленная модель:
# python -m spacy download ru_core_news_lg
nlp = spacy.load("ru_core_news_lg")

# ============================================
# ПАТТЕРНЫ ДЛЯ ГАЗЕТ
# ============================================

HEADER_PAT = re.compile(
    r"ПРОЛЕТАРИИ ВСЕХ СТРАН, СОЕДИНЯЙТЕСЬ|"
    r"ОРГАН РЕЖЕВСКОГО ГОРОДСКОГО КОМИТЕТА КПСС|"
    r"Газета основана",
    re.IGNORECASE
)

FOOTER_PAT = re.compile(
    r"АДРЕС РЕДАКЦИИ|ПИШИТЕ,\s*ЗАХОДИТЕ|ТИРАЖ|ЗВОНИТЕ",
    re.IGNORECASE
)

TV_SECTION_PAT = re.compile(
    r"НА ЭКРАНАХ ВАШИХ ТЕЛЕВИЗОРОВ|ПОНЕДЕЛЬНИК|ВТОРНИК|СРЕДА|ЧЕТВЕРГ|ПЯТНИЦА|СУББОТА|ВОСКРЕСЕНЬЕ",
    re.IGNORECASE
)

BAD_SINGLE_TOKENS = {
    "Мой", "Твой", "Нас", "Ваши", "Свой", "Наш", "Знамо",
    "Дорогу", "Съездов", "Совмина", "Таи",
    "ГАЗЕТА", "Москва", "Москбьі", "Виновников"
}

BAD_SUFFIXES = (
    "ов", "ев", "ин", "ын", "ова", "ева", "ина", "ына",
    "ский", "ская", "ские", "ских"
)

NON_PERSON_TOKENS = {
    "Правда", "Коммунизма", "ПРАВДА КОММУНИЗМА",
    "ПРОЛЕТАРИИ", "СТРАН", "ОРГАН", "КОМИТЕТА", "КПСС",
    "Совет", "Совета", "СССР", "РСФСР", "РС Ф С Р",
    "СОВЕТА НАРОДНЫХ ДЕПУТАТОВ", "НАРОДНЫХ ДЕПУТАТОВ",
    "Свердловского", "облисполкома",
    "Реж", "Режевского", "Режевский", "Режевская", "Режский",
    "Глинский", "Глинская", "Глинское",
    "Совхоз", "СОВХОЗ", "Райком", "Горком",
    "ПРОГРАММА", "Семья", "Перестройка", "ДИАЛОГ",
    "СТРАНИЦА", "РЕПОРТЕРА", "Газета",
    "Советов", "Время", "Боевой", "Комм", "Брак",
    "Своем Министерстве", "Министерстве"
}

NON_PERSON_LOWER = {
    "раци", "ферроникеля", "кормихайлович", "ёводкимоплексное",
}

BAD_CONTEXT_PATTERNS = [
    r"На\s+Мой\s+взгляд",
    r"Ваши\s+мнения",
    r"новых\s+Съездов",
    r"постановления\s+Совмина",
    r"переступает\s+Дорогу",
    r"Знамо\s+только",
]

bad_ctx_re = re.compile("|".join(BAD_CONTEXT_PATTERNS), re.IGNORECASE)

LAT_TO_CYR = str.maketrans({
    "A": "А", "a": "а",
    "B": "В", "E": "Е", "e": "е",
    "K": "К", "k": "к",
    "M": "М", "H": "Н",
    "O": "О", "o": "о",
    "P": "Р", "C": "С", "c": "с",
    "T": "Т", "X": "Х", "x": "х",
})

# ============================================
# OCR НОРМАЛИЗАЦИЯ
# ============================================

def normalize_ocr_noise(text: str) -> str:
    """
    Грубая нормализация OCR‑шума:
    - латиницу, похожую на кириллицу, маппим в кириллицу;
    - выбрасываем слова с дикой мешаниной лат/кирилл.
    """
    text = text.translate(LAT_TO_CYR)

    def clean_word(w: str) -> str:
        cyr = sum("А" <= ch <= "я" or ch in "Ёё" for ch in w)
        lat = sum("A" <= ch <= "z" or "A" <= ch <= "Z" for ch in w)
        if cyr and lat and (cyr < 2 or lat < 2):
            return ""
        return w

    return " ".join(filter(None, (clean_word(w) for w in text.split())))

# ============================================
# ПРЕПРОЦЕССИНГ ГАЗЕТ
# ============================================

def is_mostly_noise(line: str, threshold: float = 0.6) -> bool:
    if not line:
        return True
    cleaned = re.sub(r"\s+", "", line)
    if not cleaned:
        return True
    noise_chars = sum(
        1 for ch in cleaned
        if not ch.isalnum() and ch not in ".,-—:;!?\"()"
    )
    return noise_chars / len(cleaned) > threshold


def fix_hyphenation(line: str) -> str:
    return re.sub(
        r"(\b[А-ЯЁа-яё]+)-\s+([А-ЯЁа-яё]+\b)",
        r"\1\2",
        line
    )


def fix_spaced_caps(line: str) -> str:
    def repl(match):
        return match.group(0).replace(" ", "")

    return re.sub(
        r'((?:[А-ЯЁ]\s+){2,}[А-ЯЁ])',
        repl,
        line
    )


def merge_lines_gazette_style(lines: Iterable[str]) -> List[str]:
    merged = []
    prev = ""

    def is_probable_heading(line: str) -> bool:
        txt = line.strip()
        if len(txt) < 4 or len(txt) > 70:
            return False
        upp = sum(1 for ch in txt if ch.isupper())
        letters = sum(1 for ch in txt if ch.isalpha())
        if letters == 0:
            return False
        return upp / letters > 0.8

    for raw in lines:
        line = raw.rstrip()
        if not line.strip():
            if prev:
                merged.append(prev.strip())
                prev = ""
            continue

        if is_mostly_noise(line):
            continue

        line = fix_spaced_caps(line)
        line = fix_hyphenation(line)

        if not prev:
            prev = line.strip()
            continue

        if is_probable_heading(line):
            merged.append(prev.strip())
            prev = line.strip()
            continue

        if not re.search(r"[.!?…»)]\s*$", prev):
            prev = prev + " " + line.strip()
        else:
            merged.append(prev.strip())
            prev = line.strip()

    if prev:
        merged.append(prev.strip())

    return merged


def master_clean_text(raw_text: str) -> str:
    text = re.sub(r"(\w+)­\s*\n\s*(\w+)", r"\1\2", raw_text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = re.sub(r"[«»“”]", '"', text)

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        l = line.rstrip()

        if HEADER_PAT.search(l) or FOOTER_PAT.search(l) or TV_SECTION_PAT.search(l):
            continue

        if re.search(r"Индекс\s+\d+|ул\.\s*Красноармейская|Редактор|тираж", l, re.IGNORECASE):
            continue

        if is_mostly_noise(l):
            continue

        cleaned_lines.append(l)

    merged_paragraphs = merge_lines_gazette_style(cleaned_lines)
    text = "\n\n".join(merged_paragraphs)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)

    text = normalize_ocr_noise(text)

    return text.strip()


def get_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    raw_pages = [page.get_text() for page in doc]
    doc.close()
    raw = "\n".join(raw_pages)
    return master_clean_text(raw)

# ============================================
# ЭВРИСТИКИ ДЛЯ ИМЁН
# ============================================

def looks_like_glued_list(name: str) -> bool:
    """
    Сущность похожа на склеенный список имён:
    слишком много отдельных "фрагментов" с заглавной буквы.
    """
    parts = name.split()
    if len(parts) <= 4:
        return False
    cap_words = sum(1 for p in parts if p and p[0].isupper())
    return cap_words >= 3


def looks_like_garbage_name(name: str) -> bool:
    text = name.strip()
    parts = text.split()

    if looks_like_glued_list(text):
        return True

    if len(parts) == 1 and parts[0] in BAD_SINGLE_TOKENS:
        return True

    if len(parts) == 1:
        token = parts[0]
        if len(token) <= 6 and token[0].isupper() and token[1:].islower():
            return True

    if " " not in text:
        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", text)
        if len(only_cyr) >= 7:
            if not any(only_cyr.endswith(suf) for suf in BAD_SUFFIXES):
                return True

    if re.search(r"[шщжч]{3,}", text, flags=re.I):
        return True

    if re.match(r"^[а-яё]", text):
        return True

    if " " not in text and len(text) > 18:
        return True

    if re.search(r"[#|/\\\[\]\{\}<>]", text):
        return True

    if len(parts) == 1:
        word = parts[0]
        word_letters = re.sub(r"[^А-Яа-яЁё]", "", word)
        if len(word_letters) >= 6:
            vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")
            v = sum(1 for ch in word_letters if ch in vowels)
            c = len(word_letters) - v
            if v == 0 or c == 0:
                return True

    if re.search(r"[ргдлстн]{4,}", text, re.IGNORECASE) and " " not in text:
        return True

    for token in re.split(r"\s+", text):
        low = token.lower()
        if low in NON_PERSON_LOWER:
            return True

    return False


def ent_is_probably_person(ent: spacy.tokens.Span) -> bool:
    bad_pos = {"VERB", "ADV", "PRON"}
    if any(tok.pos_ in bad_pos for tok in ent):
        return False

    non_name_pos = {"PRON", "CCONJ", "SCONJ", "PART"}
    if all(tok.pos_ in non_name_pos for tok in ent):
        return False

    return True


def context_looks_like_index_list(ctx: str) -> bool:
    """
    Отфильтровывает куски, похожие на списки номеров/инициалов
    (много цифр/№/тире, мало букв).
    """
    letters = sum(ch.isalpha() for ch in ctx)
    digits = sum(ch.isdigit() for ch in ctx)
    specials = ctx.count("№") + ctx.count("N") + ctx.count("—") + ctx.count("-")
    length = len(ctx)

    if length == 0:
        return True

    if letters < 10 and (digits + specials) > 10:
        return True

    return False


def normalize_name_for_key(name: str) -> str:
    """
    Нормализация имени для дедупликации (грубо: убираем лишние пробелы).
    """
    n = re.sub(r"\.\s+", ". ", name)
    n = re.sub(r"\s{2,}", " ", n)
    return n.strip()

# ============================================
# ML-ФИЛЬТР ИМЁН
# ============================================

@dataclass
class CandidateName:
    name: str
    ctx: str
    method: str
    sent: Optional[spacy.tokens.Span] = None


class NameFilterModel:
    """
    Лёгкий бинарный классификатор поверх уже найденных имён.
    DictVectorizer + LogisticRegression.
    """

    def __init__(self):
        self.pipeline: Optional[Pipeline] = None

    def _feat(self, cand: CandidateName) -> Dict[str, Any]:
        name = cand.name
        ctx = cand.ctx

        tokens = name.split()
        n_tokens = len(tokens)
        name_len = len(name)
        has_dot = "." in name
        has_hyphen = "-" in name
        has_lower_inside = any(ch.islower() for ch in name if ch.isalpha())
        has_bad_single = (name in BAD_SINGLE_TOKENS)

        only_cyr = re.sub(r"[^А-Яа-яЁё]", "", name)
        vowels = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
        v = sum(ch in vowels for ch in only_cyr)
        c = len(only_cyr) - v

        feat = {
            "n_tokens": n_tokens,
            "name_len": name_len,
            "has_dot": has_dot,
            "has_hyphen": has_hyphen,
            "has_lower_inside": has_lower_inside,
            "has_bad_single": has_bad_single,
            "only_cyr_len": len(only_cyr),
            "vowels": v,
            "consonants": c,
            "looks_like_garbage": looks_like_garbage_name(name),
            "ctx_has_pozicion": int(bool(re.search(
                r"директор|секретарь|председатель|первый секретарь|депутат",
                ctx, re.I
            ))),
            "ctx_has_prof": int(bool(re.search(
                r"доярк|плавильщик|слесарь|редактор|журналист|учитель|врач|агроном",
                ctx, re.I
            ))),
        }

        return feat

    def fit(self, candidates: List[CandidateName], labels: List[int]) -> None:
        X = [self._feat(c) for c in candidates]
        y = labels

        vec = DictVectorizer(sparse=True)
        clf = LogisticRegression(max_iter=200, class_weight="balanced")

        self.pipeline = Pipeline([
            ("vec", vec),
            ("clf", clf),
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.pipeline.fit(X_train, y_train)

        y_pred = self.pipeline.predict(X_test)
        print("=== NameFilterModel report ===")
        print(classification_report(y_test, y_pred, digits=3))

    def predict_proba(self, candidates: List[CandidateName]) -> List[float]:
        if not self.pipeline:
            return [1.0] * len(candidates)
        X = [self._feat(c) for c in candidates]
        proba = self.pipeline.predict_proba(X)
        return [float(p[1]) for p in proba]


name_filter_model = NameFilterModel()

# ============================================
# ИЗВЛЕЧЕНИЕ ИМЁН
# ============================================

def extract_persons(
    text: str,
    use_trained_filter: bool = True,
    proba_threshold: float = 0.5
) -> List[Tuple[str, str, str]]:
    """
    Возвращает список (name, ctx, method)
    """
    results: List[CandidateName] = []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    # --- SPACY NER по абзацам ---
    for para in paragraphs:
        doc = nlp(para)
        sents = list(doc.sents)

        for ent in doc.ents:
            if ent.label_ != "PER":
                continue
            name = ent.text.strip()
            if len(name) < 3:
                continue
            if re.fullmatch(r"[А-Я]\.(\s[А-Я]\.)?", name):
                continue

            if not ent_is_probably_person(ent):
                continue

            if looks_like_garbage_name(name):
                continue

            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue

            sent = ent.sent
            if bad_ctx_re.search(sent.text):
                continue

            ctx_sents = [sent]
            try:
                idx = sents.index(sent)
                if idx + 1 < len(sents):
                    ctx_sents.append(sents[idx + 1])
            except ValueError:
                pass

            ctx_text = " ".join(s.text.strip() for s in ctx_sents)

            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method="spacy", sent=sent))

    # --- Глобальные regex‑шаблоны ---
    full_doc = nlp(text)
    full_sents = list(full_doc.sents)
    full_text = full_doc.text

    pattern_fio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_f_iio = re.compile(
        r"\b([А-ЯЁ][а-яё]+)\s+([А-ЯЁ])\.\s*([А-ЯЁ])\.\b",
        re.UNICODE
    )
    pattern_iio_f = re.compile(
        r"\b([А-ЯЁ])\.\s*([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )
    pattern_i_f = re.compile(
        r"\b([А-ЯЁ])\.\s+([А-ЯЁ][а-яё]+)\b",
        re.UNICODE
    )

    def add_regex_matches(pattern, method_label: str):
        nonlocal results
        for m in pattern.finditer(full_text):
            name = m.group(0).strip()
            if len(name) < 4:
                continue
            if any(tok in name for tok in NON_PERSON_TOKENS):
                continue
            if looks_like_garbage_name(name):
                continue

            start, end = m.span()
            sent_idx = None
            for idx, s in enumerate(full_sents):
                if s.start_char <= start < s.end_char:
                    sent_idx = idx
                    break

            if sent_idx is not None:
                ctx_sents = [full_sents[sent_idx]]
                if sent_idx + 1 < len(full_sents):
                    ctx_sents.append(full_sents[sent_idx + 1])
                ctx_text = " ".join(s.text.strip() for s in ctx_sents)
                sent = full_sents[sent_idx]
            else:
                ctx_start = max(0, start - 80)
                ctx_end = min(len(full_text), end + 200)
                ctx_text = full_text[ctx_start:ctx_end].strip()
                sent = None

            if bad_ctx_re.search(ctx_text):
                continue
            if context_looks_like_index_list(ctx_text):
                continue

            results.append(CandidateName(name=name, ctx=ctx_text, method=method_label, sent=sent))

    add_regex_matches(pattern_fio, "regex_fio")
    add_regex_matches(pattern_f_iio, "regex_f_iio")
    add_regex_matches(pattern_iio_f, "regex_iio_f")
    add_regex_matches(pattern_i_f, "regex_i_f")

    # --- "Подписи" в виде «А. Фамилия» ---
    signature_pattern = re.compile(
        r"^\s*([А-ЯЁ])\.\s*([А-ЯЁ][а-яё]+)\b.*$",
        re.MULTILINE
    )

    for para_idx, para in enumerate(paragraphs):
        for m in signature_pattern.finditer(para):
            short_name = f"{m.group(1)}. {m.group(2)}"
            if looks_like_garbage_name(short_name):
                continue
            prev_para = paragraphs[para_idx - 1] if para_idx > 0 else ""
            ctx_text = (prev_para + "\n\n" + para).strip()
            if context_looks_like_index_list(ctx_text):
                continue
            results.append(CandidateName(name=short_name, ctx=ctx_text, method="signature", sent=None))

    # --- Убираем дубликаты по (name, усечённый контекст) ---
    unique: Dict[Tuple[str, str], CandidateName] = {}
    for cand in results:
        clean_name = re.sub(r"\.$", "", cand.name).strip()
        clean_name = normalize_name_for_key(clean_name)
        key = (clean_name, cand.ctx[:120])
        if key not in unique:
            unique[key] = CandidateName(name=clean_name, ctx=cand.ctx, method=cand.method, sent=cand.sent)

    final_candidates = list(unique.values())

    # --- Финальный ML‑фильтр ---
    if use_trained_filter:
        proba = name_filter_model.predict_proba(final_candidates)
        filtered = [
            (cand.name, cand.ctx, cand.method)
            for cand, p in zip(final_candidates, proba)
            if p >= proba_threshold
        ]
    else:
        filtered = [(c.name, c.ctx, c.method) for c in final_candidates]

    return filtered

# ============================================
# ОБУЧЕНИЕ ФИЛЬТРА ПО РАЗМЕЧЕННОМУ JSON
# ============================================

def load_labeled_candidates(json_path: str) -> Tuple[List[CandidateName], List[int]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cands: List[CandidateName] = []
    labels: List[int] = []

    for item in data:
        cands.append(CandidateName(
            name=item["name"],
            ctx=item["ctx"],
            method=item.get("method", "unknown"),
            sent=None
        ))
        labels.append(int(item["label"]))

    return cands, labels


def train_name_filter_from_json(json_path: str) -> None:
    global name_filter_model
    cands, labels = load_labeled_candidates(json_path)
    name_filter_model.fit(cands, labels)

# ============================================
# ПАРСИНГ ИМЕНИ ГАЗЕТЫ И ДАТЫ ИЗ ИМЕНИ ФАЙЛА
# (ЗАГЛУШКА, ПРИ НЕОБХОДИМОСТИ ПЕРЕДЕЛАЙ ПОД СВОЙ ФОРМАТ)
# ============================================

def parse_newspaper_meta_from_filename(filename: str) -> Tuple[str, str]:
    """
    Пример: 'Pravda_1985-01-05_issue12.pdf' ->
      newspaper_name='Pravda', publication_date='1985-01-05'
    Подстрой под свой реальный формат.
    """
    base = os.path.splitext(filename)[0]
    # пробуем найти yyyy-mm-dd
    m = re.search(r"\d{4}-\d{2}-\d{2}", base)
    if m:
        publication_date = m.group(0)
        newspaper_name = base[:m.start()].rstrip("_ -")
    else:
        publication_date = "UNKNOWN"
        newspaper_name = base
    return newspaper_name, publication_date

# ============================================
# ПРИМЕР ЗАПУСКА
# ============================================

if __name__ == "__main__":
    # 0. Создаём БД (один раз при старте)
    create_database()

    # 1. Путь к размеченному JSON
    labeled_json = r"C:\Users\nikol\Desktop\disser\labeled_names.json"  # поправь при необходимости

    if os.path.exists(labeled_json):
        train_name_filter_from_json(labeled_json)
    else:
        print(f"Внимание: файл разметки не найден: {labeled_json}")
        print("Фильтр имён будет работать только на эвристиках (без ML).")

    # 2. Обработка папки с pdf-газетами
    folder = r"C:\Users\nikol\Desktop\disser\newspapers"

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        pdf_path = os.path.join(folder, fname)
        print(f"\n=== Обработка файла: {fname} ===")
        try:
            full_text = get_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Ошибка чтения PDF: {e}")
            continue

        persons = extract_persons(full_text, use_trained_filter=True, proba_threshold=0.5)
        print(f"Найдено уникальных имён: {len(persons)}")
        for name, ctx, method in persons[:20]:
            snippet = ctx[:180].replace("\n", " ")
            print(f"[{method}] {name} :: {snippet}{'...' if len(ctx) > 180 else ''}")
            print("-" * 60)

        # 3. Сохранение в базу
        newspaper_name, publication_date = parse_newspaper_meta_from_filename(fname)
        insert_mentions(
            newspaper_name=newspaper_name,
            publication_date=publication_date,
            pdf_filename=fname,
            persons=persons
        )

База данных 'newspapers.db' и таблица 'newspaper_mentions' созданы.
Внимание: файл разметки не найден: C:\Users\nikol\Desktop\disser\labeled_names.json
Фильтр имён будет работать только на эвристиках (без ML).

=== Обработка файла: 1990-01-06_Правда коммунизма 1990 003.pdf ===
Найдено уникальных имён: 80
[spacy] Вавилов :: окружной избирательной комиссии Алапаевского территориального округа № 660 По выборам народных депутатов РСФ СР. 3 ее работе приняли участие представители Режа В. Пальцев — зам ест...
------------------------------------------------------------
[spacy] Н. Кислицына :: окружной избирательной комиссии Алапаевского территориального округа № 660 По выборам народных депутатов РСФ СР. 3 ее работе приняли участие представители Режа В. Пальцев — зам ест...
------------------------------------------------------------
[spacy] Соловьева :: Соловьева — от коллектива никелевого завода. Е. Щ ербаков — от коллектива механического завода.
--------------------------------------------